# 8단계 한눈에 보기

이 노트북은 작업하며 주고받은 답변(✍️ 칸과 클로드의 답변)을 빼고, 코드 · 결과 · 물음마다의 결론(📌)만 남긴 판이다. 단계마다 한 일과 정한 것은 README 「전체 흐름」에 있다.

### 말 뜻 — 표를 읽을 때 쓰는 말 (10-28에 통일함)

| 말 | 뜻 |
|---|---|
| 실제 | 그해(주로 2026년)에 실제로 발전한 양 · 실제 값입니다. 전력거래소 자료 그대로입니다 |
| 모델 예측 | 2023 ~ 2025년을 배운 모델이 그때의 날씨 · 시각 · 날 종류를 보고 예측한 양입니다. 9단계까지는 「평소라면」이라고 불렀습니다 |
| 실제 − 모델 예측 | 발전량 표의 숫자입니다. + 면 실제가 모델 예측보다 많았다, − 면 적었다는 뜻입니다 |
| 평소 해 | 2023 · 2024 · 2025년입니다. 「평소」는 이 세 해만 가리킵니다 |
| 평소 범위 | 평소 해에 모델 예측이 한 달 평균으로 가장 크게 틀린 크기(±2,177 MWh)입니다. 모델이 예측해 본 2024 · 2025년으로 냈습니다. 2026년의 차이가 이 안이면 「달랐다고 말할 근거가 없다」로 읽습니다 |
| 세 해 평균 | 2023 ~ 2025년 같은 달 값의 평균입니다. 모델 없이 계산한 것으로, 10단계 가격 표(2-가 · 2-라)에서만 씁니다 |

앞 단계 글에서 발전량을 두고 쓴 「평소보다 많았다 · 적었다」는 「모델 예측보다 많았다 · 적었다」로 읽습니다.


# 08. 모델 확정과 핵심인자 — 전체 11단계 중 8단계

**이 단계의 목적** — 7단계에서 기준선보다 낫다고 확인한 모델 넷 가운데 9단계에서 2026년을 맞힐 모델 하나를 고르고, 어느 변수가 LNG를 얼마나 움직였는지(핵심인자)를 적는 단계입니다. 그래야 「왜 이 모델인가」와 「LNG를 움직인 것은 무엇이었나」에 숫자로 답하고, 1단계에 짐작해 둔 원인과 맞춰 볼 수 있습니다.

## 데이터와 모델

**데이터 흐름**

1. 2·3단계 — 시간마다 LNG·기온·햇빛 같은 값을 받아서, 빈 곳을 고친 표를 만들었습니다.
2. 4단계 — 무엇이 LNG를 움직이는지 봤습니다. 몇 시인가 · 무슨 날인가 · 기온 · 햇빛이 컸습니다.
3. 5단계 — 그것들을 모델이 쓸 수 있는 열로 바꿔서 파일 하나(`data/hot/features_2023_2024_2025.csv`)로 저장했습니다.
4. 6단계 — 「2023년으로 배워 2024년을 맞혀 보기」로 연습하면서, 무엇을 넣고 무엇과 비교하고 무엇이 나오면 잘된 것으로 볼지를 정했습니다.
5. 7단계 — 같은 파일로 모델 넷을 2023·2024년으로 배워, 처음 보는 해인 2025년을 한 번 맞혀 채점했습니다. 어디서 어떻게 틀렸는지도 봤습니다.
6. 8단계(이 노트북) — 같은 파일을 읽고, 7단계와 같은 코드로 모델 넷을 다시 만듭니다(준비 ③). 그 넷 가운데 하나를 고르고, 어느 변수가 LNG를 움직였는지 적습니다.

**모델의 입력** (6단계에서 정한 그대로)

- 몇 시인가 · 몇 월인가
- 무슨 날인가 — 월요일 · 화~금 · 토 · 일 · 공휴일 · 명절
- 기온 — 선형회귀는 16도보다 추운 만큼과 24도보다 더운 만큼, 트리 계열은 기온 그대로
- 3시간 동안 기온이 바뀐 만큼 — 추운 쪽과 더운 쪽 (선형회귀에만)
- 햇빛 세기(일사)
- 쉬는 날이면서 몇 시인가 (선형회귀에만)
- 넣지 않는 것 — 원전 · 습도 · 석탄 · 총 전력거래량 · 가격

**모델 넷** (6단계 물음 6에서 정함)

- 선형회귀 — 열마다 기울기 하나를 배웁니다. 「16도보다 1도 더 추우면 LNG가 몇 MWh 는다」처럼 바로 읽을 수 있습니다.
- 기온 × 시간대 선형회귀 — 위와 같은 선형회귀인데, 기온의 기울기를 시간대마다 따로 배웁니다.
- 랜덤포레스트 — 「몇 시 · 몇 도 · 무슨 날이면 LNG가 이만큼」으로 나누는 의사결정나무를 300개 따로 만들고, 그 예측을 평균 냅니다.
- 그래디언트 부스팅 — 의사결정나무를 하나씩 더해 가면서, 앞의 나무들이 틀린 만큼을 다음 나무가 줄이도록 배웁니다.

**7단계에서 정한 것**

- 물음 1(채점하기 전 점검) — 모델 넷을 6단계 계획대로 만들었고, 열끼리 겹치지 않았습니다. 잔차가 이웃한 시각끼리 같은 쪽으로 틀려서(독립성이 깨짐), 모델은 그대로 두고 숫자를 읽는 방법을 바꿨습니다. 그래서 8단계에서는 p값으로 변수를 고르지 않고, 선형회귀 기울기의 p값은 4주 떨어진 시각까지 감안해 다시 낸 표준오차로 읽습니다.
- 물음 2(2025년 채점) — 모델 넷 모두 2025년에도 기준선(2023·2024년의 같은 달 · 같은 시각 · 같은 날 종류의 평균)보다 잘 맞혔습니다. 넷 다 쓰고, 하나를 고르는 것은 8단계로 넘겼습니다.
- 물음 3(어디서 어떻게 틀렸나) — 모델 넷과 기준선이 같은 곳에서 같은 쪽으로 틀렸습니다. 까닭은 대부분 이 프로젝트가 일부러 모델 밖에 둔 것(늘어난 태양광 설비 · 원전 · 석탄 · 수요)이라 모델은 고치지 않았고, 8단계에서도 새 변수를 넣지 않습니다.
- 물음 4(인과지도) — 그 까닭을 인과지도에 「모델이 못 본 고리」로 더했습니다. 인과지도는 8단계에서 쓰지 않습니다(지침 — 계산에 쓰이지 않는 그림이라서).

**8단계의 입력과 결과**

| 순서 | 입력 | 처리 | 결과 | 용도 |
|---|---|---|---|---|
| 1 | 5단계가 저장한 표 하나 — 2023~2025년 시각마다 LNG · 기온 · 일사 · 몇 시 · 몇 월 · 무슨 날 | 준비 ②가 읽고, 2025년 LNG는 바로 비운다. 7단계와 같은 열을 만든다 | 모델마다 넣을 열 | 아래 2~5 |
| 2 | 2023년 · 2023·2024년의 열과 그 시각의 LNG | 준비 ③이 7단계와 같은 코드로 모델 넷을 두 번 배운다 — 2023년으로 배워 2024년 맞히기(6단계의 연습) · 2023·2024년으로 배워 2025년 맞히기(7단계의 채점) | 두 해의 시각마다 맞힌 LNG와 실제 LNG | 물음 1. 물음 2는 준비 ③의 모델 설정만 쓰고, 2025년 LNG는 쓰지 않는다 |
| 3 | 두 해의 맞힌 LNG와 실제 LNG | 1-가에서 정한 「고르는 순서」를 차례로 대고, 볼트 순서로 고른 것과 나란히 놓는다 | 최종 모델 하나와 고른 까닭 | 9단계에서 2026년을 맞히는 모델 |
| 4 | 모델 넷 · 4단계 탐색에서 본 것 | 변수마다 여덟 칸(역할 · 형태 · 제외 사유 · 탐색적 기법 결과 · 모델별 중요도 · 기술적 검토 · 총점 · 선정 사유)을 채운다 | 핵심인자 목록 | 물음 3 · 11단계 결론 |
| 5 | 핵심인자 목록 · 1단계에 적은 원인 목록 여섯 줄 | 나란히 놓는다 | 원인 목록에 있었는데 안 나온 것 · 없었는데 나온 것 · 방향이 반대인 것 | 1단계로 되돌아갈지 · 11단계 |

**먼저 준비 ① ~ ③을 돌립니다.** 준비 ③은 모델 넷을 두 번 배웁니다.

| 단계 | 이름 |
|---|---|
| 1 | 과제 정의 |
| 2 | 데이터 수집과 구성 보기 |
| 3 | 정제 |
| 4 | 탐색적 분석 |
| 5 | 변환·파생변수 |
| 6 | 분석계획 |
| 7 | 모델링과 검정 |
| 8 | 모델 확정과 핵심인자 ◀ 이 노트북 |
| 9 | 이 과제만의 물음 — 2026년은 평소와 얼마나 달랐나 |
| 10 | 그 답을 숫자로 내기 |
| 11 | 결론 |

진행 기록(지금 어디)은 리포의 `README.md`가 원본입니다. 단계마다 무엇을 하고 내가 무엇을 정하는지는 옵시디언 볼트 「빅데이터 분석의 순서」에 있습니다.

그 밖의 표지 글(물음마다 정한 것 · 물음 셋 · 교차검증을 여기서는 어떻게 하나 · 앞 단계에서 넘어온 것 등)은 맨 아래 「8단계 표지 — 자세한 내용」에 있습니다.

In [97]:
# 8단계 모델 확정과 핵심인자 · 준비 ①
import os                                      # 폴더와 파일 경로를 다루는 기본 모듈
import numpy as np                             # 숫자 계산
import pandas as pd                            # 표(데이터프레임)를 다루는 라이브러리
import matplotlib.pyplot as plt                # 그림을 그리는 라이브러리
import sklearn                                 # 판 번호를 적어 두려고 부른다
from IPython.display import display            # 표를 글자가 아니라 격자로 그린다
from sklearn.linear_model import LinearRegression                                   # 선형회귀 — 7단계와 같다
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor   # 트리 계열 둘 — 7단계와 같다
plt.rcParams["font.family"] = "AppleGothic"    # 맥에서 그림 속 한글이 깨지지 않게
plt.rcParams["axes.unicode_minus"] = False     # 그림의 마이너스 부호가 깨지지 않게
pd.set_option("display.max_columns", 50)       # 열이 많아도 생략하지 않고 다 보이게

ROOT = os.getcwd()                             # 지금 폴더(보통 notebooks/)에서 시작해서
while not os.path.isdir(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)               # data 폴더가 보일 때까지 위로 = 프로젝트 루트
HOT = os.path.join(ROOT, "data", "hot")        # 5단계가 만들어 둔 파일이 있는 폴더

print("프로젝트 루트:", ROOT)                    # 경로가 맞는지만 확인한다
print("scikit-learn 판:", sklearn.__version__)   # 그래디언트 부스팅은 판이 다르면 소수점 아래 넷째 숫자가 달라질 수 있어 적어 둔다(7단계 준비 ①)

프로젝트 루트: /Users/kimchangju/vs코드 /개인 프로젝트/lng-project
scikit-learn 판: 1.9.0


In [98]:
# 8단계 모델 확정과 핵심인자 · 준비 ②
tbl = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig")  # 5단계 「저장」 셀이 만든 표 — 6 · 7단계와 같은 파일
tbl["ts"] = pd.to_datetime(tbl["ts"])            # 글자로 저장된 시각을 다시 시각 자료형으로
tbl = tbl.set_index("ts").sort_index()           # 시각을 줄 이름으로 두고 시간 순서대로 놓는다
tbl.loc[tbl["year"] == 2025, "lng_mwh"] = np.nan   # 누수를 막는다 — 2025년 LNG는 읽자마자 비우고, 준비 ③ 하나만 다시 읽는다

전체 = tbl.dropna(subset=["3시간 전 기온"])          # 3시간 전 기온이 빈 9시각(2023-01-01 · 2024-03-01 · 2024-09-01의 0~2시)은 뺀다 — 6 · 7단계와 같다
y = 전체["lng_mwh"]                                # 맞힐 값 — 그 시각의 LNG 발전량(MWh). 2025년은 비어 있다

def 선형열(d):                                     # 선형회귀에 넣는 열 — 7단계 준비 ②와 같은 67열
    시각열 = pd.get_dummies(d["hour24"].astype(str), prefix="시", drop_first=True).astype(float)   # 1~23시 0/1 열 — 0시가 기준
    월열 = pd.get_dummies(d["month"].astype(str), prefix="월", drop_first=True).astype(float)      # 2~12월 0/1 열 — 1월이 기준
    날열 = pd.get_dummies(d["날 종류"], prefix="날").astype(float).drop(columns="날_월")            # 화~금·토·일·공휴일·명절 — 월요일이 기준
    쉬는날 = d["날 종류"].isin(["토", "일", "공휴일", "명절"]).astype(float)                         # 쉬는 날이면 1
    쉬는날시간대 = (시각열 - 1 / 24).mul(쉬는날, axis=0)                                            # 5단계 물음 7 — 시간대 열의 평균(1/24)을 빼고 쉬는 날과 곱한다
    쉬는날시간대.columns = [f"쉬는 날×{c[2:]}시" for c in 시각열.columns]                            # 열 이름 — 「쉬는 날×9시」
    기상 = pd.DataFrame({"16도보다 추운 만큼": d["16도보다 추운 만큼"],                               # 5단계 물음 1 — 16·24도
                         "24도보다 더운 만큼": d["24도보다 더운 만큼"],
                         "3시간 동안 바뀐 만큼 (추운 쪽)": d["16도보다 추운 만큼"] - d["3시간 전 16도보다 추운 만큼"],   # 6단계 4-라 — 지금 값 − 3시간 전 값
                         "3시간 동안 바뀐 만큼 (더운 쪽)": d["24도보다 더운 만큼"] - d["3시간 전 24도보다 더운 만큼"],
                         "일사": d["icsr"]})                                                       # 5단계 물음 3 — 일사 한 열
    return pd.concat([시각열, 월열, 날열, 기상, 쉬는날시간대], axis=1)

기온두열 = ["16도보다 추운 만큼", "24도보다 더운 만큼"]                                               # 선형회귀의 지금 기온 두 열
def 시간대기온열(d):                                # 기온 × 시간대 선형회귀 — 기온 두 열 대신 시간대마다 나눈 48열, 113열(6단계 물음 6)
    나눔 = pd.DataFrame({f"{c}×{h}시": d[c] * (d["hour24"] == h) for c in 기온두열 for h in range(24)}, index=d.index)   # 그 시간대일 때만 값, 아니면 0
    return 선형열(d).drop(columns=기온두열).join(나눔)

def 트리열(d):                                     # 트리 계열에 넣는 열 — 숫자 그대로 7열(6단계 닫는 칸 ② 「입력 — 트리 계열」)
    return d[["hour24", "month", "icsr", "ta", "dow", "is_holiday", "명절"]]

입력 = {"선형회귀": 선형열(전체), "기온 × 시간대 선형회귀": 시간대기온열(전체), "트리 계열": 트리열(전체)}   # 세 해의 열을 한 번에 만든다 — 해마다 같은 열이 되게

print(f"표 모양: {len(tbl):,}행 · {tbl.shape[1] + 1}열 — 6 · 7단계가 읽은 것과 같은 파일")
쓰는곳 = {2023: "배우는 해 — 두 번 모두", 2024: "한 번은 맞히는 해, 한 번은 배우는 해", 2025: "맞히는 해 — LNG는 준비 ③에서 한 번 꺼낸다"}
display(pd.DataFrame({"시각 수": 전체.groupby("year").size(), "LNG가 있는 시각": y.notna().groupby(전체["year"]).sum(),
                      "이 노트북에서": pd.Series(쓰는곳)}).rename_axis("해"))

표 모양: 25,536행 · 25열 — 6 · 7단계가 읽은 것과 같은 파일


,시각 수,LNG가 있는 시각,이 노트북에서
해,,,
2023,8757,8757,배우는 해 — 두 번 모두
2024,8010,8010,"한 번은 맞히는 해, 한 번은 배우는 해"
2025,8760,0,맞히는 해 — LNG는 준비 ③에서 한 번 꺼낸다


In [99]:
# 8단계 모델 확정과 핵심인자 · 준비 ③
모델들 = {"선형회귀": (lambda: LinearRegression(), "선형회귀"),                                     # 7단계 2-가와 같은 넷 — 6단계 물음 6에서 정한 모델과 설정
          "기온 × 시간대 선형회귀": (lambda: LinearRegression(), "기온 × 시간대 선형회귀"),
          "랜덤포레스트": (lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=1, max_features=0.5,
                                                      n_jobs=-1, random_state=0), "트리 계열"),
          "그래디언트 부스팅": (lambda: HistGradientBoostingRegressor(early_stopping=False, random_state=0), "트리 계열")}
기준선이름 = "기준선(달·시각·날별 평균)"                                                         # 6단계 물음 2에서 가장 나았던 기준선
쉬는날 = tbl["날 종류"].isin(["토", "일", "공휴일", "명절"])                                       # 7단계와 같은 쉬는 날

원본 = pd.read_csv(os.path.join(HOT, "features_2023_2024_2025.csv"), encoding="utf-8-sig", usecols=["ts", "lng_mwh"])   # 2025년 LNG를 꺼낸다 — 이 노트북에서 여기 한 곳(7단계 2-가와 같은 줄)
원본["ts"] = pd.to_datetime(원본["ts"])

def 기준선(배운해, 시):                                                                        # 배운 해의 같은 달 · 같은 시각 · 같은 날 종류 평균으로 맞힌다(7단계 2-가 · 3-너와 같은 방법)
    yb = tbl.loc[tbl["year"].isin(배운해), "lng_mwh"]
    평균 = yb.groupby([tbl.loc[yb.index, "month"], tbl.loc[yb.index, "hour24"], 쉬는날.loc[yb.index]]).mean()
    키 = pd.MultiIndex.from_arrays([tbl.loc[시, "month"], tbl.loc[시, "hour24"], 쉬는날.loc[시]])
    return pd.Series(평균.reindex(키).values, index=시)

두번 = {}                                                                                      # 「앞 해로 배워 다음 해 맞히기」 두 번 — 맞힌 해: (실제, {방법: 예측})
for 맞힌해, 배운해 in [(2024, [2023]), (2025, [2023, 2024])]:                                   # 2023 → 2024(6단계의 연습) · 2023·2024 → 2025(7단계의 채점)
    배울곳 = 전체["year"].isin(배운해)
    맞힐곳 = 전체["year"] == 맞힌해
    시 = 전체.index[맞힐곳]                                                                     # 맞히는 해의 시각
    실제 = y[맞힐곳] if 맞힌해 == 2024 else 원본.set_index("ts")["lng_mwh"].reindex(시)          # 2024년은 준비 ②의 LNG, 2025년은 위에서 꺼낸 LNG
    예측 = {기준선이름: 기준선(배운해, 시)}
    for 이름, (만들기, 열) in 모델들.items():                                                   # 모델 넷 — 배운 해로 배우고, 맞히는 해는 입력만으로 맞힌다
        X = 입력[열]
        예측[이름] = pd.Series(만들기().fit(X[배울곳], y[배울곳]).predict(X[맞힐곳]), index=시)
    두번[f"{맞힌해}년"] = (실제, 예측)

def R2(실, 예): return 1 - ((실 - 예) ** 2).sum() / ((실 - 실.mean()) ** 2).sum()                  # 결정계수 — 6 · 7단계와 같은 식
칠단계 = {"2024년": {"선형회귀": 0.7524, "기온 × 시간대 선형회귀": 0.7679, "랜덤포레스트": 0.7701, "그래디언트 부스팅": 0.7807},   # 7단계 1-가 [표 1] 「이 노트북 결정계수」
          "2025년": {"선형회귀": 0.7007, "기온 × 시간대 선형회귀": 0.7280, "랜덤포레스트": 0.7427, "그래디언트 부스팅": 0.7553}}  # 7단계 2-가 [표 1]
for 해, (실, 예) in 두번.items():
    같다 = all(f"{R2(실, 예[m]):.4f}" == f"{v:.4f}" for m, v in 칠단계[해].items())             # 7단계에서 나온 결정계수와 소수점 넷째 자리까지 같은가
    배운 = "2023년" if 해 == "2024년" else "2023·2024년"
    print(f"{배운}으로 배워 {해} {len(실):,}시각을 맞혔다 — 모델 넷의 결정계수가 7단계와 {'같다' if 같다 else '다르다(scikit-learn 판을 확인해 주세요)'}")

2023년으로 배워 2024년 8,010시각을 맞혔다 — 모델 넷의 결정계수가 7단계와 같다
2023·2024년으로 배워 2025년 8,760시각을 맞혔다 — 모델 넷의 결정계수가 7단계와 같다


# 물음 1 — 모델 넷 가운데 어느 것으로 9단계에 가나

<div style="border: 2px solid var(--vscode-focusBorder, #3794ff); border-left-width: 6px; border-radius: 8px; padding: 4px 18px 10px; background: var(--vscode-textBlockQuote-background, rgba(127, 127, 127, 0.1));">

## 📌 물음 1 결론

**왜 생긴 물음인가** — 7단계에서 모델 넷(선형회귀 · 기온 × 시간대 선형회귀 · 랜덤포레스트 · 그래디언트 부스팅)이 모두 2025년에도 기준선보다 잘 맞혀서, 넷 가운데 9단계에서 2026년 1 ~ 6월을 맞힐 모델 하나를 고르는 일을 8단계로 넘겼습니다(7단계 2-다 「✍️ 읽고 나서」 ① 「높은 것, 즉 현실적으로 예측력이 높고 현실을 잘 반영하는 것을 골라야겠지??」). 그 하나를 무엇으로 어떻게 고를지 정하고 고른 물음입니다.

### 결정 — 9단계에서 2026년을 맞힐 모델은 그래디언트 부스팅이다. 1-나 ~ 1-마 결과를 보기 전에 정한 두 가지 고르는 방법이 모두 이 모델을 골랐다

**같이 정한 것**

1. 고르는 방법은 둘이다. B방식으로 정하고, A방식은 나란히 놓고 본다. B방식은 세 가지를 차례로 본다 — 기준선(배운 해에서 같은 달 · 같은 시각 · 쉬는 날인지가 같은 시각의 평균)보다 못한 달이 되풀이되나, 기온 · 햇빛 · 시간대 · 날을 4단계와 거꾸로 배운 것이 있나, 남은 모델 가운데 두 해 모두 더 잘 맞히고 나눠 봐도 꾸준히 나은가. A방식은 교재 순서대로 2025년 결정계수가 가장 높은 모델이다(1-가 「✍️ 읽고 나서」 「b방식으로 가되, … a방식 경우와 b경우 두 가지 다 하면 안되나??」).
2. B방식의 첫째는 덜 엄격하게 센다 — 같은 달이 2024년과 2025년 두 해 모두 기준선보다 더 틀렸을 때만 뺀다(1-가 「✍️ 다시 읽고 나서 (2)」 「모델링 학습 범위 자체가 좁아서,, 조금 덜 엄격하게 봐도 좋지 않을까??」). 1-나 결과를 보기 전에 적으셨고, 클로드는 결과를 본 뒤에 동의했다. 한 달이라도 더 틀린 달을 다 세는 쪽으로 했어도 그래디언트 부스팅만 남았다.
3. 두 방식이 다른 모델을 고르면 둘 다 9단계에 가져가고 결론은 B방식 모델로 쓴다(1-가 「✍️ 다시 읽고 나서」 ① 「나도 일이 크게 늘지 않으면 이렇게 하는게 적합하게 느껴져」). 이번에는 같은 모델이라 하나로 간다. 그래서 6단계 물음 5에서 정한 「고른 모델을 같은 열 · 같은 설정으로 2023 ~ 2025년 세 해에 다시 배워 2026년을 맞힌다」는 그대로다.
4. B방식 셋째에서 남은 두 모델 가운데 어느 쪽이 나은지는 결정계수 차이 0.002로 가르지 않고, 두 해 모두 더 잘 맞히고 나눠 본 63묶음(달 · 시간대 · 기온 · 요일과 날 종류로 나눈 것)의 절반 넘게 덜 틀리는지로 본다(클로드 제안 — 0.002는 5단계 1-마에서 냉방 기준 온도를 23 ~ 27도로 옮길 때 생긴 결정계수 차이 0.502 − 0.500에서 꺼낸 숫자라서).

**결정 근거**

- B방식 첫째 — 같은 달이 두 해 모두 기준선보다 더 틀린 달은 선형회귀 7 · 11월, 랜덤포레스트 12월이었고, 기온 × 시간대 선형회귀와 그래디언트 부스팅은 없었다(1-나 [표 2]).
- B방식 둘째 — 넷 모두 다섯 가지를 4단계와 같은 쪽으로 배웠다 — 추울수록 · 더울수록 LNG를 더 맞히고, 낮에 햇빛이 강할수록 덜 맞히고, 3시보다 18시에 더 맞히고, 수요일보다 일요일에 덜 맞혔다(1-마 [표 2]).
- B방식 셋째 — 그래디언트 부스팅이 기온 × 시간대 선형회귀보다 두 해 모두 결정계수가 높고(0.7807 · 0.7553 대 0.7679 · 0.7280) 평균 절대 오차가 작았고(2,004 · 2,364 대 2,072 · 2,503 MWh), 63묶음 가운데 52묶음에서 덜 틀렸다(1-다 · 1-라).
- A방식 — 2025년 결정계수 0.7553으로 1등, 2등 랜덤포레스트 0.7427과 0.0126 차이(1-다 [표 1]).
- 판단을 적으신 칸 — 1-마 「✍️ 읽고 나서」 ① 「지속적으로 그래디언트가 가장 좋은 결과를 보이고 있긴 한데.. 일단 그래야하지 않을까?」.

**남은 것**

- 고르는 방법은 7단계에서 2025년 결과를 본 뒤에 세웠다. 그래디언트 부스팅의 2025년 성적(결정계수 0.7553)은 처음 보는 해의 성적보다 조금 좋게 나온 값일 수 있다 — 11단계 한계.
- 그래디언트 부스팅이 다른 모델보다 더 틀린 묶음이 있다 — 랜덤포레스트보다 9묶음, 기온 × 시간대 선형회귀보다 11묶음(1-라). 예를 들어 2025년 10월 · 11월. 9단계에서 2026년의 그런 묶음은 조심해서 읽는다.
- 기울기를 숫자로 읽는 선형회귀를 옆에 둘지는 물음 2(어느 변수가 LNG를 얼마나 움직였나)와 9단계에서 정한다. 7단계에서 넘어온 「3시간 동안 바뀐 만큼 (더운 쪽)」 점검(기울기가 0과 구별되지 않은 선형회귀의 열)도 그때 본다 — 그래디언트 부스팅은 이 열을 쓰지 않는다.
- 모델 넷을 평균 내서 맞히는 방법은 해 보지 않았다 — 11단계 한계에 「해 보지 않은 것」으로 적는다.

</div>

## 1-가 — 넷 가운데 하나를 고를 때 무엇이 맞아야 하나 (1-나부터의 표를 보기 전에)

**이 셀이 답하는 물음** — 「현실적으로 예측력이 높고 현실을 잘 반영하는 것」을 고르려면, 무엇이 맞아야 하나.

**세 기준 — 앞에서 이미 정한 것**

| 기준 | 무엇을 보나 | 무엇으로 보나 | 어디서 정했나 |
|---|---|---|---|
| 설명력 | 처음 보는 해를 얼마나 맞히나 | 2025년 결정계수와 평균 절대 오차 | 7단계 물음 2 결론 칸 「같이 정한 것」 4 |
| 안정성 | 해가 바뀌어도, 나눠 봐도 같은 순서인가 | 「앞 해로 배워 다음 해 맞히기」 두 번(2023 → 2024 · 2023·2024 → 2025)과, 그것을 달 · 시간 묶음 · 기온 묶음 · 요일로 나눠 본 것 | 6단계 물음 5 · 7단계 물음 2 결론 칸 |
| 효율성 | 같은 성능이면 더 단순한가 | 넣는 변수가 더 적은 쪽. 배우고 맞히는 데 걸리는 시간도 같이 적는다(시간은 클로드가 더함) | 볼트 8단계 절 · 7단계 물음 2 결론 칸 |

- 모델끼리 「같은 성능」인지를 무엇으로 볼지는 아래 1-가에서 정했습니다 — 두 번(2024 · 2025년을 맞힌 것) 모두 앞서고 나눠 본 줄에서도 더 자주 덜 틀리는가(B방식). 결정계수 차이 0.002(5단계 1-마에서 클로드가 꺼낸 숫자, 6단계에서 열을 넣을지와 모델이 기준선보다 나은지를 가를 때 쓰심)로 고르는 방식은 A방식으로 나란히 놓습니다.

**아직 정하지 않은 것 셋** — 아래 칸에 적습니다. 읽어도 어떻게 정할지 모르겠으면 「클로드 생각 먼저」라고만 적으셔도 됩니다. 그러면 제 생각을 먼저 드리고, 그것을 보고 정하시면 됩니다.

## 1-나 — 달마다 나눠 보면 모델이 기준선보다 더 틀린 달이 있나 (2024 · 2025년)

**이 셀을 돌리기 전에** 1-가 「✍️ 다시 읽고 나서 (2)」 ②(이 표를 얼마나 엄격하게 셀지)를 먼저 적습니다.

**이 셀이 답하는 물음** — 「고르는 순서」 1(먼저 맞아야 하는 조건): 2024년 열한 달 · 2025년 열두 달 가운데, 기준선보다 평균 절대 오차가 큰 달이 있는 모델은 어느 것인가.

**표의 숫자** — 단위는 MWh(한 시각 평균)입니다. [표 1]의 칸은 그 달 모델의 평균 절대 오차에서 같은 달 기준선의 평균 절대 오차를 뺀 값입니다. +면 그 달 모델이 기준선보다 더 틀렸고, −면 덜 틀렸습니다. 모델끼리 비교하는 표가 아니라, 모델마다 기준선과 비교하는 표입니다. [표 2]는 [표 1]에서 +인 달을 모델마다 센 것이고, 맨 오른쪽 칸은 같은 달이 두 해 모두 +인 달의 수입니다(②에서 「덜 엄격하게」를 고르시면 이 칸으로 봅니다).

In [100]:
# 8단계 모델 확정과 핵심인자 · 1-나
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 두 해를 맞힌 값이 준비 ③에서 만들어집니다"
모델이름 = list(모델들)                                                         # 선형회귀 · 기온 × 시간대 선형회귀 · 랜덤포레스트 · 그래디언트 부스팅
해이름 = {"2024년": "2024년을 맞힘 (2023년으로 배워서)", "2025년": "2025년을 맞힘 (2023 · 2024년으로 배워서)"}   # 표에 쓰는 해 이름

차이 = {}                                                                       # (해, 모델) → 달마다 (모델의 평균 절대 오차 − 기준선의 평균 절대 오차)
for 해, (실, 예) in 두번.items():
    달 = 실.index.month                                                          # 그 시각이 몇 월인가
    기준 = (실 - 예[기준선이름]).abs().groupby(달).mean()                        # 기준선이 그 달 한 시각에 평균 몇 MWh 틀렸나
    for m in 모델이름:
        차이[(해이름[해], m)] = (실 - 예[m]).abs().groupby(달).mean() - 기준       # +면 그 달 모델이 기준선보다 더 틀렸다

표1 = pd.DataFrame(차이).reindex(range(1, 13))                                   # 열두 달 — 2024년 8월은 자료가 없어 빈칸이 된다
표1.index = [f"{d}월" for d in 표1.index]
표1.index.name = "달"
print("[표 1] 달마다 — 모델의 평균 절대 오차 − 기준선의 평균 절대 오차 (MWh, 한 시각 평균)")
print("        +면 그 달 모델이 기준선보다 더 틀렸고, −면 덜 틀렸다 · 모델마다 기준선과 비교하는 표다")
display(표1.map(lambda v: "자료 없음" if pd.isna(v) else f"{v:+,.0f}"))

앞해, 뒤해 = 해이름["2024년"], 해이름["2025년"]
표2 = pd.DataFrame({"2024년 (열한 달 가운데)": {m: int((표1[(앞해, m)] > 0).sum()) for m in 모델이름},     # 2024년에 +인 달
                    "2025년 (열두 달 가운데)": {m: int((표1[(뒤해, m)] > 0).sum()) for m in 모델이름}})    # 2025년에 +인 달
표2["두 해 합 (스물세 달 가운데)"] = 표2.sum(axis=1)                                                   # 「그대로」로 셀 때 — 한 달이라도 +면 걸린다
표2["같은 달이 두 해 모두 + (열한 달 가운데)"] = [int(((표1[(앞해, m)] > 0) & (표1[(뒤해, m)] > 0)).sum()) for m in 모델이름]   # 「덜 엄격하게」로 셀 때 — 8월은 2024년이 없어 세지 않는다
표2.index.name = "모델"
print("[표 2] 모델마다 기준선보다 더 틀린 달의 수 — [표 1]에서 +인 달을 센 것")
display(표2)

[표 1] 달마다 — 모델의 평균 절대 오차 − 기준선의 평균 절대 오차 (MWh, 한 시각 평균)
        +면 그 달 모델이 기준선보다 더 틀렸고, −면 덜 틀렸다 · 모델마다 기준선과 비교하는 표다


2024년을 맞힘 (2023년으로 배워서)                                 \
                       선형회귀 기온 × 시간대 선형회귀 랜덤포레스트 그래디언트 부스팅   
달                                                            
1월                      -87          -191   -507      -618   
2월                     -555          -574   -404      -485   
3월                     -434          -466   -481      -631   
4월                     +138           +45    -70       -93   
5월                      +47           -65   -177      -100   
6월                     -137          -252   -253      -263   
7월                     +141           -80    -68      -143   
8월                    자료 없음         자료 없음  자료 없음     자료 없음   
9월                     -987          -990   -686      -876   
10월                    -328          -326   -256      -277   
11월                     +50            +3   -344      -161   
12월                     -45          -111     +6       -95   

    2025년을 맞힘 (2023 · 2024년으로 배워서)                                 
                              선형회귀 기온 × 시간대 선형회귀 랜덤포레스트 그래디언트 부스팅  
달                                                                  
1월                             +38           -56    -61      -150  
2월                            -596          -611   -635      -826  
3월                             +37           -43   -261      -369  
4월                            -209          -314   -454      -523  
5월                            -197          -326   -400      -413  
6월                             +65           -71   -140      -220  
7월                            +470          -133   -174      -218  
8월                            +211          -181   -291      -316  
9월                            -411          -366   -793      -799  
10월                           -418          -474   -254      -257  
11월                            +36           -60   -323      -260  
12월                            +72           -38   +118       -10

[표 2] 모델마다 기준선보다 더 틀린 달의 수 — [표 1]에서 +인 달을 센 것


,2024년 (열한 달 가운데),2025년 (열두 달 가운데),두 해 합 (스물세 달 가운데),같은 달이 두 해 모두 + (열한 달 가운데)
모델,,,,
선형회귀,4,7,11,2
기온 × 시간대 선형회귀,2,0,2,0
랜덤포레스트,1,1,2,1
그래디언트 부스팅,0,0,0,0


## 1-다 — 두 번(2024 · 2025년을 맞힌 것)의 결정계수와 평균 절대 오차

**이 셀이 답하는 물음** — 처음 보는 해를 맞혔을 때, 기준선과 모델 넷이 얼마나 맞혔나. 「고르는 순서」 3의 앞 절반(두 번 모두 결정계수가 높고 평균 절대 오차가 작은가)과, A방식의 첫 기준(2025년 결정계수가 가장 높은 모델이 그다음 모델보다 0.002 넘게 높은가)에 쓰는 숫자입니다.

**표의 숫자** — [표 1] 결정계수는 1에 가까울수록 잘 맞힌 것입니다. [표 2] 평균 절대 오차는 한 시각에 평균 몇 MWh 틀렸나(+와 −를 떼고)라서 작을수록 덜 틀린 것입니다. 두 표 모두 모델끼리 비교하는 표이고, 줄(방법)은 7단계 2-가와 같은 순서입니다.

In [101]:
# 8단계 모델 확정과 핵심인자 · 1-다
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 두 해를 맞힌 값이 준비 ③에서 만들어집니다"
해이름 = {"2024년": "2024년을 맞힘 (2023년으로 배워서)", "2025년": "2025년을 맞힘 (2023 · 2024년으로 배워서)"}   # 1-나와 같은 해 이름
방법들 = [기준선이름] + list(모델들)                                             # 기준선과 모델 넷 — 7단계 2-가와 같은 줄 순서

결정 = pd.DataFrame({해이름[해]: {b: R2(실, 예[b]) for b in 방법들} for 해, (실, 예) in 두번.items()})   # 준비 ③과 같은 결정계수 식
결정.index.name = "방법"
print("[표 1] 결정계수 — 처음 보는 해를 맞혔을 때 · 1에 가까울수록 잘 맞혔다")
display(결정.map(lambda v: f"{v:.4f}"))

절대 = pd.DataFrame({해이름[해]: {b: (실 - 예[b]).abs().mean() for b in 방법들} for 해, (실, 예) in 두번.items()})   # 한 시각에 평균 몇 MWh 틀렸나
절대.index.name = "방법"
print("[표 2] 평균 절대 오차 — 한 시각에 평균 몇 MWh 틀렸나(+와 −를 떼고, MWh) · 작을수록 덜 틀렸다")
display(절대.map(lambda v: f"{v:,.0f}"))

[표 1] 결정계수 — 처음 보는 해를 맞혔을 때 · 1에 가까울수록 잘 맞혔다


,2024년을 맞힘 (2023년으로 배워서),2025년을 맞힘 (2023 · 2024년으로 배워서)
방법,,
기준선(달·시각·날별 평균),0.6945,0.6678
선형회귀,0.7524,0.7007
기온 × 시간대 선형회귀,0.7679,0.7280
랜덤포레스트,0.7701,0.7427
그래디언트 부스팅,0.7807,0.7553


[표 2] 평균 절대 오차 — 한 시각에 평균 몇 MWh 틀렸나(+와 −를 떼고, MWh) · 작을수록 덜 틀렸다


,2024년을 맞힘 (2023년으로 배워서),2025년을 맞힘 (2023 · 2024년으로 배워서)
방법,,
기준선(달·시각·날별 평균),"2,342","2,723"
선형회귀,"2,146","2,652"
기온 × 시간대 선형회귀,"2,072","2,503"
랜덤포레스트,"2,049","2,421"
그래디언트 부스팅,"2,004","2,364"


## 1-라 — 나눠 본 줄에서 두 모델씩, 어느 쪽이 더 자주 덜 틀리나 (2024 · 2025년)

**이 셀이 답하는 물음** — 해가 바뀌어도, 나눠 봐도 모델끼리 순서가 같은가(안정성). 「고르는 순서」 3의 뒤 절반(나눠 본 줄에서도 절반이 넘는 줄을 덜 틀리는가)과, A방식의 안정성(2025년 결정계수의 차이가 0.002 안쪽일 때)에 쓰는 숫자입니다.

**표의 숫자** — [표 1]의 칸은 「줄 이름의 모델이 칸 이름의 모델보다 평균 절대 오차가 작았던 줄 수 / 나눠 본 줄 수」입니다. 예를 들어 어떤 칸이 「40 / 63」이면(지어낸 숫자), 63줄 가운데 40줄에서 줄 이름의 모델이 칸 이름의 모델보다 덜 틀렸다는 뜻입니다. 모델끼리 비교하는 표입니다. [표 2]는 나눠 본 줄이 해마다 · 가지마다 몇 개인지입니다.

In [102]:
# 8단계 모델 확정과 핵심인자 · 1-라
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 두 해를 맞힌 값이 준비 ③에서 만들어집니다"
모델이름 = list(모델들)                                                         # 모델 넷 — 1-나와 같은 순서
요일이름 = ["월", "화", "수", "목", "금", "토", "일"]

def 나눈줄(해):                                                                  # 7단계 3-다 · 3-바 · 3-카와 같게 나눈 줄마다, 모델 넷의 평균 절대 오차
    실, 예 = 두번[해]
    오차 = pd.DataFrame({m: (실 - 예[m]).abs() for m in 모델이름})                  # 시각마다 틀린 크기(+와 −를 뗀다)
    t = 오차.index
    달 = pd.Series([f"{d}월" for d in t.month], index=t)                         # 달
    시간 = pd.Series(pd.cut(t.hour, bins=[-1, 5, 9, 15, 19, 23],
                           labels=["0~5시 (밤)", "6~9시", "10~15시 (낮)", "16~19시", "20~23시"]).astype(str), index=t)   # 3-다와 같은 시간 묶음
    기온 = pd.cut(전체.loc[t, "ta"], bins=[-np.inf, -5, 0, 16, 24, 30, np.inf],
                 labels=["영하 5도 이하", "영하 5도 ~ 0도", "0 ~ 16도", "16 ~ 24도", "24 ~ 30도", "30도 넘음"]).astype(str)   # 3-바와 같은 기온 묶음
    날 = pd.Series([요일이름[d] for d in t.dayofweek], index=t)                   # 요일
    날 = 날.where(~전체.loc[t, "날 종류"].isin(["공휴일", "명절"]), 전체.loc[t, "날 종류"])   # 공휴일 · 명절은 따로 — 3-카와 같다
    줄들 = []
    for 가지, 묶음 in [("달", 달), ("시간 묶음", 시간), ("기온 묶음", 기온), ("요일과 날 종류", 날)]:
        g = 오차.groupby(묶음).mean()                                             # 그 묶음에 든 시각의 평균 절대 오차
        g.index = pd.MultiIndex.from_product([[해], [가지], g.index])
        줄들.append(g)
    return pd.concat(줄들)

줄 = pd.concat([나눈줄("2024년"), 나눈줄("2025년")])                                  # 두 해의 나눈 줄 — 줄마다 모델 넷의 평균 절대 오차

이김 = pd.DataFrame(index=모델이름, columns=모델이름, dtype=object)
for a in 모델이름:
    for b in 모델이름:
        이김.loc[a, b] = "—" if a == b else f"{int((줄[a] < 줄[b]).sum())} / {len(줄)}"   # 줄 이름의 모델이 칸 이름의 모델보다 덜 틀린 줄 수
이김.index.name = "이 모델이 ↓ · 이 모델보다 →"
print(f"[표 1] 두 모델씩 — 줄 이름의 모델이 칸 이름의 모델보다 평균 절대 오차가 작았던 줄 수 / 나눠 본 줄 {len(줄)}개")
print("        모델끼리 비교하는 표다 · 한 칸과 대각선 건너편 칸을 더하면 나눠 본 줄 수가 된다")
display(이김)

수 = 줄.groupby(level=[0, 1]).size().unstack(0).reindex(["달", "시간 묶음", "기온 묶음", "요일과 날 종류"])   # 해 · 가지마다 줄 수
수.loc["합"] = 수.sum()
수.index.name = "나눈 가지"
print("[표 2] 나눠 본 줄이 몇 개인가 — 해마다 · 가지마다")
display(수)

[표 1] 두 모델씩 — 줄 이름의 모델이 칸 이름의 모델보다 평균 절대 오차가 작았던 줄 수 / 나눠 본 줄 63개
        모델끼리 비교하는 표다 · 한 칸과 대각선 건너편 칸을 더하면 나눠 본 줄 수가 된다


,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅
이 모델이 ↓ · 이 모델보다 →,,,,
선형회귀,—,3 / 63,9 / 63,6 / 63
기온 × 시간대 선형회귀,60 / 63,—,20 / 63,11 / 63
랜덤포레스트,54 / 63,43 / 63,—,9 / 63
그래디언트 부스팅,57 / 63,52 / 63,54 / 63,—


[표 2] 나눠 본 줄이 몇 개인가 — 해마다 · 가지마다


,2024년,2025년
나눈 가지,,
달,11,12
시간 묶음,5,5
기온 묶음,6,6
요일과 날 종류,9,9
합,31,32


## 1-마 — 한 가지만 바꾸면 모델이 맞히는 LNG가 4단계에서 본 쪽으로 움직이나 (배운 방향)

**이 셀이 답하는 물음** — 「고르는 순서」 2(먼저 맞아야 하는 조건): 모델 넷이 기온 · 햇빛 · 시간대 · 날 종류를 4단계에서 본 것과 같은 쪽으로 배웠나. 1-가 「✍️ 내 답변」 ②에 적으신 「특정한 방향, 영역에서 잘못된 해석이 명확하다면, 결정계수를 기준으로만 보는건 부족하지 않나?」를 보는 셀입니다.

**어떻게 보나.** 배운 두 해의 시각을 그대로 두고 한 가지만 바꿔 모델 넷으로 다시 맞힌 뒤, 맞힌 값의 평균이 얼마나 움직였는지 봅니다. 다른 것은 모두 그대로라, 그 한 가지를 바꾼 만큼만 움직입니다. 바꾸는 폭은 흔한 범위(배운 두 해 시각의 양 끝 5%씩을 뺀 범위, 1-가에서 정함) 안입니다.

| 바꾸는 것 | 어떻게 바꾸나 | 4단계에서 본 쪽 — 결과를 보기 전에 적음 |
|---|---|---|
| 기온 — 추운 쪽 | 모든 시각을 16도 → 추운 쪽 끝(아래 5%)으로. 3시간 전 기온도 같이 바꿔, 3시간 동안 바뀐 만큼은 0 | + (추울수록 LNG가 는다 — 4단계 09-13 「기온은 18~20도가 바닥인 U자」, 5단계 1-나 바닥 16도) |
| 기온 — 더운 쪽 | 모든 시각을 24도 → 더운 쪽 끝(위 5%)으로 | + (더울수록 LNG가 는다 — 같은 U자) |
| 낮의 햇빛 | 낮 10 ~ 15시 시각만, 일사를 약한 쪽 끝 → 강한 쪽 끝(낮 시각의 아래 · 위 5%)으로 | − (낮에 태양광이 많을수록 LNG가 준다 — 4단계 물음 2 결론) |
| 시간대 | 모든 시각을 3시 → 18시로. 일사는 둘 다 0 | + (수요가 느는 저녁을 LNG가 맡는다 — 4단계 물음 2 「같이 정한 것」 ①, 3시 → 18시) |
| 날 | 모든 시각을 수요일 → 일요일로(공휴일 · 명절은 아님) | − (쉬는 날 LNG가 적다 — 4단계 09-13 주말 상관 −0.33) |

**표의 숫자** — [표 1]은 흔한 범위의 양 끝 값입니다. [표 2]의 칸은 「바꾼 뒤 맞힌 LNG의 평균 − 바꾸기 전 맞힌 LNG의 평균」(MWh, 한 시각 평균)이고, 옆에 위 표의 쪽과 같으면 「같다」, 다르면 「반대」를 적었습니다. +면 바꾼 뒤에 LNG를 더 맞힌 것입니다. 모델끼리 크기를 비교하는 표가 아니라, 모델마다 쪽이 맞는지 보는 표입니다.

In [103]:
# 8단계 모델 확정과 핵심인자 · 1-마
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 모델 넷의 설정과 열이 준비 ② · ③에서 만들어집니다"
배울곳 = 전체["year"].isin([2023, 2024])                                        # 2025년을 맞힌 모델과 같은 두 해로 배운다
틀 = 전체[배울곳].copy()                                                          # 배운 두 해의 시각 — 여기서 한 가지만 바꿔 다시 맞혀 본다
다시 = {이름: 만들기().fit(입력[열][배울곳], y[배울곳]) for 이름, (만들기, 열) in 모델들.items()}   # 준비 ③의 2025년 모델과 같은 설정 · 같은 random_state

def 평균예측(d):                                                                 # d의 시각을 모델 넷으로 맞힌 값의 평균(한 시각 MWh)
    둘 = pd.concat([d, 틀], ignore_index=True)                                    # 원래 시각을 뒤에 붙여야 0/1 열이 배울 때와 같은 열로 만들어진다
    n = len(d)
    열들 = {"선형회귀": 선형열(둘), "기온 × 시간대 선형회귀": 시간대기온열(둘), "트리 계열": 트리열(둘)}
    return {이름: float(다시[이름].predict(열들[열].iloc[:n][입력[열].columns]).mean()) for 이름, (_, 열) in 모델들.items()}

def 기온을(d, g):                                                                # 지금 기온과 3시간 전 기온을 모두 g도로 — 3시간 동안 바뀐 만큼은 0
    d = d.copy()
    d["ta"] = g
    d["3시간 전 기온"] = g
    d["16도보다 추운 만큼"] = d["3시간 전 16도보다 추운 만큼"] = max(16 - g, 0)
    d["24도보다 더운 만큼"] = d["3시간 전 24도보다 더운 만큼"] = max(g - 24, 0)
    return d

def 일사를(d, g):                                                                # 일사만 g로
    d = d.copy()
    d["icsr"] = g
    return d

def 시간을(d, h):                                                                # 모든 시각을 h시로, 일사는 0으로 — 해가 없을 때의 시간대 차이만 본다
    d = d.copy()
    d["hour24"] = h
    d["icsr"] = 0.0
    return d

def 날을(d, 종류):                                                               # 모든 시각을 수요일(평일) 또는 일요일로 — 공휴일 · 명절은 아니다
    d = d.copy()
    d["날 종류"] = 종류
    d["dow"] = 2 if 종류 == "화~금" else 6                                       # 2 = 수요일, 6 = 일요일
    d["is_holiday"] = 0
    d["명절"] = 0
    return d

낮 = 틀["hour24"].between(10, 15)                                                # 낮 10 ~ 15시
기온끝 = 틀["ta"].quantile([0.05, 0.95])                                          # 흔한 범위 — 배운 두 해 시각의 양 끝 5%씩을 뺀다
일사끝 = 틀.loc[낮, "icsr"].quantile([0.05, 0.95])                                # 낮 시각의 일사도 같은 방법으로

바꿔본것 = {                                                                       # 이름: (바꾸기 전, 바꾼 뒤, 4단계에서 본 쪽)
    "기온 16도 → 추운 쪽 끝 (4단계: +)": (기온을(틀, 16), 기온을(틀, 기온끝[0.05]), +1),
    "기온 24도 → 더운 쪽 끝 (4단계: +)": (기온을(틀, 24), 기온을(틀, 기온끝[0.95]), +1),
    "낮 일사 약한 끝 → 강한 끝 (4단계: −)": (일사를(틀[낮], 일사끝[0.05]), 일사를(틀[낮], 일사끝[0.95]), -1),
    "3시 → 18시, 일사 0 (4단계: +)": (시간을(틀, 3), 시간을(틀, 18), +1),
    "수요일 → 일요일 (4단계: −)": (날을(틀, "화~금"), 날을(틀, "일"), -1),
}
움직임 = {}
for 이름, (앞d, 뒤d, 쪽) in 바꿔본것.items():
    앞, 뒤 = 평균예측(앞d), 평균예측(뒤d)                                           # 바꾸기 전 · 바꾼 뒤 맞힌 LNG의 평균
    움직임[이름] = {m: f"{뒤[m] - 앞[m]:+,.0f} · " + ("같다" if np.sign(뒤[m] - 앞[m]) == 쪽 else "반대") for m in 모델들}

print("[표 1] 흔한 범위의 양 끝 — 배운 두 해(2023 · 2024년) 시각 가운데 아래 5% · 위 5%가 시작되는 값")
display(pd.DataFrame({"아래 끝 (5%)": [f"{기온끝[0.05]:.1f}", f"{일사끝[0.05]:.2f}"], "위 끝 (95%)": [f"{기온끝[0.95]:.1f}", f"{일사끝[0.95]:.2f}"]},
                     index=["기온 (도)", "낮 10 ~ 15시 일사 (MJ/m²)"]).rename_axis("무엇"))
print("[표 2] 한 가지만 바꿨을 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — 바꾼 뒤 − 바꾸기 전")
print("        +면 바꾼 뒤에 LNG를 더 맞혔다 · 「같다」면 칸 이름의 괄호(4단계에서 본 쪽)와 같은 쪽이다 · 모델마다 쪽을 보는 표다")
display(pd.DataFrame(움직임).rename_axis("모델"))

[표 1] 흔한 범위의 양 끝 — 배운 두 해(2023 · 2024년) 시각 가운데 아래 5% · 위 5%가 시작되는 값


,아래 끝 (5%),위 끝 (95%)
무엇,,
기온 (도),-3.2,29.1
낮 10 ~ 15시 일사 (MJ/m²),0.21,3.28


[표 2] 한 가지만 바꿨을 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — 바꾼 뒤 − 바꾸기 전
        +면 바꾼 뒤에 LNG를 더 맞혔다 · 「같다」면 칸 이름의 괄호(4단계에서 본 쪽)와 같은 쪽이다 · 모델마다 쪽을 보는 표다


,기온 16도 → 추운 쪽 끝 (4단계: +),기온 24도 → 더운 쪽 끝 (4단계: +),낮 일사 약한 끝 → 강한 끝 (4단계: −),"3시 → 18시, 일사 0 (4단계: +)",수요일 → 일요일 (4단계: −)
모델,,,,,
선형회귀,"+5,401 · 같다","+4,985 · 같다","-7,995 · 같다","+10,534 · 같다","-5,235 · 같다"
기온 × 시간대 선형회귀,"+5,046 · 같다","+2,994 · 같다","-7,476 · 같다","+10,894 · 같다","-5,249 · 같다"
랜덤포레스트,"+3,776 · 같다","+2,779 · 같다","-6,229 · 같다","+9,008 · 같다","-5,098 · 같다"
그래디언트 부스팅,"+4,112 · 같다","+2,722 · 같다","-6,903 · 같다","+9,163 · 같다","-5,201 · 같다"


# 물음 2 — 어느 변수가 LNG를 얼마나 움직였나 (핵심인자)

<div style="border: 2px solid var(--vscode-focusBorder, #3794ff); border-left-width: 6px; border-radius: 8px; padding: 4px 18px 10px; background: var(--vscode-textBlockQuote-background, rgba(127, 127, 127, 0.1));">

## 📌 물음 2 결론

**왜 생긴 물음인가** — 8단계는 9단계에 가져갈 모델 하나와 핵심인자를 정하는 단계입니다. 물음 1에서 모델을 그래디언트 부스팅으로 정했고, 물음 2는 모델에 넣은 다섯 변수(몇 시인가 · 무슨 날인가 · 기온 · 햇빛 · 몇 월인가)가 LNG를 얼마나 움직였는지 봅니다. 창주 님 말로는 「디스케일링해가지고 어느 정도의 비중을 변수마다 갖고 있는지 보고, 순서를 결정하는 것」입니다(2-바 「✍️ 내 답변」 3).

### 결정 — 모델에 넣은 다섯 모두 핵심인자이고, 몇 시인가 → 무슨 날인가 → 기온 → 햇빛 → 몇 월인가 순서로 LNG를 움직였다

**같이 정한 것**

1. 순서는 다섯 변수를 모두 밤낮 모든 시각으로 계산한 뒤섞어 보기(2-다)의 것이다. 바꿔 넣어 보기(2-나)는 햇빛만 낮 10 ~ 15시로 계산해서 햇빛이 2번째였다(2-마).
2. 순서는 모델 넷(선형회귀 둘 · 트리 계열 둘)의 평균으로 매긴다 — 한 모델의 중요도로 정하지 않는다(2-가, 교재). 그래서 선형회귀도 핵심인자의 순서에 들어갔다.
3. 「햇빛은 해가 있는 시각에만 LNG를 움직인다」와 「기온과 햇빛은 크기가 비슷해서 모델에 따라 순서가 바뀐다」를 붙인다(2-마).
4. 결론 문장에 크기를 같이 적는다 — 몇 시인가는 다음인 무슨 날인가의 두 배쯤이고, 몇 월인가는 몇 시인가의 10분의 1쯤이다(클로드 판단 — 정하신 뒤에 붙임).
5. 범위 — 다섯은 LNG를 움직이는 것 전부가 아니라 모델에 넣은 것 가운데 LNG를 움직인 것이다. 원전 · 석탄 · 가격처럼 LNG를 움직이는 다른 것은 일부러 모델 밖에 두었고, 7단계에서 2025년에 크게 틀린 날의 틀린 양도 대부분 원전 · 석탄 발전량이 평소와 달랐던 만큼이었다(7단계 3-거). 모델에 더하는 것은 없다 — 2025년 결과를 본 뒤라 더하면 그 채점이 처음 보는 해의 성적이 아니게 된다(클로드 제안 — 2-바 「✍️ 다시 읽고 나서」 1 · 2).
6. 기온 · 햇빛이 한 칸(1도 · 1 MJ/m²)에 LNG를 얼마나 움직이나(효과 크기)는 2-사에서 내고, 11단계에서 인과지도 화살표 옆과 결론 글에 옮겨 적는다. 이 숫자로 순서는 바뀌지 않는다 — 단위가 붙은 숫자라 변수끼리 비교하지 않는다(클로드 판단에 「좋아」(10-16 채팅) — 그 문답은 2-바 「✍️ 다시 읽고 나서」 3 ~ 7).

**결론 문장** — 2-가에서 정한 대로 둘로 나눠 적는다. 모델이 숫자로 보여 준 것을 먼저 쓰고, 왜 그런지 풀어 쓴 해석은 따로 적는다.

- 모델이 보여 준 것 — 모델에 넣은 다섯 가운데 LNG를 맞히는 데 가장 크게 쓰인 것은 몇 시인가이고, 무슨 날인가 · 기온 · 햇빛 · 몇 월인가 순서였다. 그 변수만 뒤섞으면 한 시각에 몇 시인가 2,923 · 무슨 날인가 1,418 · 기온 1,002 · 햇빛 896 · 몇 월인가 280 MWh를 더 틀렸다 — 몇 시인가는 무슨 날인가의 두 배쯤, 몇 월인가는 몇 시인가의 10분의 1쯤이다. 기온과 햇빛은 크기가 비슷해서 모델에 따라 순서가 바뀌고, 햇빛은 해가 있는 시각에만 LNG를 움직인다. 기온 · 햇빛이 한 칸 움직일 때는(2-사 [표 1], 모델 넷의 평균) 16도보다 1도 추울 때마다 LNG가 한 시각에 239 MWh, 24도보다 1도 더울 때마다 661 MWh 늘고, 낮에 일사가 1 MJ/m² 강할 때마다 2,329 MWh 준다. 이 셋은 단위가 달라 서로 크기를 비교하지 않는다 — 흔히 움직이는 폭을 곱하면 낮 햇빛 7,150 · 추운 쪽 기온 4,584로, 바꿔 넣어 보기의 폭(7,150 · 4,601)과 거의 같다(2-사 [표 2]).
- 해석 — 모델이 보여 준 것이 아니라 풀어 쓴 것이다. 전력 수요 · 태양광 발전량 · 원전 정비 일정은 모델에 넣지 않았다. 몇 시인가 — 하루의 생활과 일의 흐름에 따라 전력 수요가 오르내린다. 무슨 날인가 — 쉬는 날 · 연휴에는 사회 활동이 줄어 전력 수요가 준다. 기온 — 16도보다 추우면 난방, 24도보다 더우면 냉방으로 전력 수요가 늘고, 추우면 열병합 발전소의 발전량도 는다. 햇빛 — 일사가 강할수록 전력시장의 태양광 발전량이 늘어 LNG가 준다. 몇 월인가 — 기온 · 일사를 따로 넣어서, 달마다 다른 원전 정비 · 석탄 상한 같은 나머지가 남는다(2-바 표의 「역할」 줄).

**결정 근거** — 2-다 [표 1] 「넷의 평균」(그 변수만 뒤섞었을 때 늘어난 평균 절대 오차, 한 시각 MWh) — 몇 시인가 2,923 · 무슨 날인가 1,418 · 기온 1,002 · 햇빛 896 · 몇 월인가 280. 바꿔 넣어 보기(2-나 [표 1])에서도 몇 시인가가 가장 크고 몇 월인가가 가장 작았다. 모델 없이 묶어 본 것(2-라)은 다른 변수가 섞여서 순서에 쓰지 않았다 — 몇 월인가는 모델 없이 5,423이었지만, 날씨를 그대로 두고 달만 바꾸면 2,104였다(2-라 [표 3], 그래디언트 부스팅). 정하신 말은 2-바 「✍️ 읽고 나서」 — 「너가 제시한 순서대로 1~5번 순서대로 가겠네」. 「1도에 얼마」는 2-사 [표 1]의 「넷의 평균」과 [표 2]에서 가져왔다.

**앞 단계에서 넘어온 것 가운데 다른 방법으로 대신한 것** (클로드 판단)

- 선형회귀 기울기를 열끼리 비교할 때 쓰기로 한 표준화 기울기(기울기 × 배운 기간의 표준편차, 5단계 물음 6) — 따로 계산하지 않았다. 바꿔 넣어 보기(2-나)와 뒤섞어 보기(2-다)가 다섯 변수를 모델 넷 모두에서 LNG의 MWh 하나로 놓고 비교해서 같은 일을 했고, 2-사 [표 2]에서 한 칸에 얼마에 흔히 움직이는 폭을 곱한 것이 표준화 기울기와 같은 생각이다.
- 쉬는 날 × 시간대 열로 토 · 일 기울기를 읽는 것(5단계 7-다) — 열마다 읽지 않았다. 2-나 · 2-다에서 무슨 날인가를 바꾸거나 뒤섞으면 그 날 종류로 만든 쉬는 날 × 시간대 열도 같이 바뀌어서(준비 ②의 선형회귀 열), 무슨 날인가 전체가 LNG를 얼마나 움직였는지로 봤다.
- 트리 계열은 겹치는 열끼리 중요도를 나눠 가져서 하나하나가 작게 보일 수 있다는 것(6단계 4-다) — 2-다에서 한 변수에서 나온 열(무슨 날인가 넷 · 기온 여섯)을 같은 순서로 함께 뒤섞어서, 열마다 중요도가 나뉘지 않게 했다.

**넣지 않은 것과 그 까닭(제외 사유)**

| 넣지 않은 것 | 까닭 | 정한 곳 |
|---|---|---|
| 원전 발전량 | 2023 · 2024년 두 해 자료로는 원전의 효과를 석탄 변화와 떼어 낼 수 없었다. 2024년에 석탄이 줄면서 원전과 LNG가 같이 늘어, 넣으면 배운 방향이 거꾸로였고 2024년 결정계수도 내려갔다 | 6단계 물음 1 |
| 습도 — 그대로 넣기 · 습하면 1로 넣기 · 불쾌지수 75 넘는 만큼 | 2023년으로 배워 2024년을 맞힌 결정계수가 −0.0032 · −0.0017 · +0.0012로, 셋 다 문턱(0.002 넘게 오르나)을 넘지 못했다. 7단계에서 2025년 습한 날에 예측이 작았지만, 그 차이는 해가 강한 한낮에 가장 커서 대부분 흐린 날 태양광 쪽으로 읽었다(7단계 3-사 · 3-차) | 6단계 물음 4 |
| 트리 계열의 3시간 전 기온 | 넣으면 2024년 결정계수가 랜덤포레스트 +0.0011 · 그래디언트 부스팅 +0.0009라 문턱을 넘지 못했다. 선형회귀에는 「3시간 동안 바뀐 만큼」으로 넣었다 | 6단계 4-마 |
| 30도에서 한 번 더 나누는 기온 열 | 5단계에서 넣어 봤지만 결정계수가 0.001만 올랐다. 7단계에서 2025년 30도 넘는 시각에 가장 크게 틀렸지만(모델 넷 모두 예측이 실제보다 컸다), 2025년을 본 뒤에 넣으면 그 채점이 처음 보는 해의 성적이 아니게 된다(7단계 3-바) | 5단계 1-사 |
| 태양광 설비용량 | 해마다 늘어서, 배운 두 해 뒤의 값은 모두 배운 범위 밖이었다. 햇빛 세기(일사) 한 열로 바꿨다 | 3단계 · 5단계 물음 3 |
| 석탄 발전량 | 같은 시각에 LNG와 함께 정해지고 2026년에 달라진 것 자체라, 넣으면 확인하려는 2026년의 차이가 사라진다 | 설계(지침) |
| 총 전력수요 | 넣으면 LNG = 총량 − 나머지라는 뺄셈이 되어 모델이 필요 없어진다 | 설계(지침) |
| 가격 | 2026년에 달라진 것이 가격이라, 넣으면 「평소와 얼마나 달랐나」가 0에 가까워진다. 10단계 해석에 쓴다 | 설계(지침) |

**한계**

1. 결론의 순서를 매기는 방법을 결과를 본 뒤에 골랐다 — 2-가에서 결과를 보기 전에 정한 기준(바꿔 넣어 보기)대로라면 햇빛이 2번째였다(2-마).
2. 뒤섞어 보기도 새벽 2시에 한낮의 햇빛 값이 붙은 것처럼 실제로는 없는 시각을 만든다(2-마).
3. 몇 시인가와 몇 월인가에는 여러 까닭이 같이 들어 있다 — 몇 시인가에는 저녁에 전기를 많이 쓰는 것과 해가 지면 태양광이 끊기는 것이, 몇 월인가에는 원전 정비 · 석탄 상한처럼 모델에 넣지 않은 것이 같이 들어 있다. 결론에는 모델이 숫자로 보여 준 것을 먼저 쓰고, 왜 그런지 풀어 쓴 해석은 따로 적는다(2-가).
4. 선형회귀의 「3시간 동안 바뀐 만큼(더운 쪽)」 열은 기울기가 0과 구별되지 않았다(p값 0.89). 고른 그래디언트 부스팅은 이 열을 쓰지 않아서 그대로 둔다(2-바 기술적 검토).
5. 「1도에 얼마」는 흔한 범위 끝까지 움직인 양을 바꾼 크기로 나눈 평균이다 — 추운 쪽은 16도에서 영하 3.2도까지, 더운 쪽은 24도에서 29.1도까지, 햇빛은 낮 일사 0.21에서 3.28 MJ/m²까지. 선형회귀는 1도마다 같지만, 트리 계열은 기온마다 다를 수 있다. 예를 들어 영하 쪽의 1도와 15도 쪽의 1도가 같지 않을 수 있다(2-사, 클로드 판단).

**남은 것** — 물음 3(1단계에 적은 원인 후보와 핵심인자를 맞춰 보기). 「1도에 얼마」는 11단계에서 인과지도 화살표 옆과 결론 글에 옮겨 적는다. 이 물음에서 2025 · 2026년 LNG는 쓰지 않았다.

</div>

## 2-가 — 무엇이 맞으면 핵심인자라고 부를까 (2-나부터의 표를 보기 전에)

**이 셀이 답하는 물음** — 모델에 넣은 변수 가운데 「LNG를 움직였다」고 말할 것을 고르려면, 무엇이 맞아야 하나.

**여덟 칸 — 변수마다 무엇을 적나**

| 칸 | 무엇을 적나 | 어디서 채우나 |
|---|---|---|
| 역할 | 그 변수가 어떤 길로 LNG를 움직인다고 봤나 | 1단계 원인 목록 · 6단계 인과지도 |
| 형태 | 모델에 어떤 숫자로 넣었나 | 5 · 6단계 |
| 제외 사유 | 넣지 않은 변수는 왜 뺐나 | 6단계 물음 1 · 4 |
| 탐색적 기법 결과 | 모델 없이 본 것 — 상관 · 시간대별 평균 같은 것 | 4단계 → 2-라 |
| 모델별 중요도 | 모델마다 그 변수가 LNG를 얼마나 움직였나, 그 변수를 뒤섞으면 얼마나 더 틀리나 | 2-나 · 2-다 |
| 기술적 검토 | 그 숫자를 믿을 수 있나 — 겹치는 열 · p값 | 7단계 → 2-바 |
| 총점 | 위 칸들을 모은 점수 | 2-나의 폭을 모델 넷에서 평균 낸 순서(2-가에서 정함). 2-다 · 2-라와 크게 다르면 따로 적는다 |
| 선정 사유 | 핵심인자로 골랐나, 왜 | 2-바 「✍️ 내 답변」 |

**정할 것 둘** — 아래 칸에 적습니다. 어떻게 정할지 모르겠으면 「클로드 생각 먼저」라고만 적으셔도 됩니다.

## 2-나 — 변수마다 LNG를 얼마나 움직였나 (흔히 있는 값을 다 넣어 봤을 때의 폭, 모델 넷)

**이 셀이 답하는 물음** — 2-가 ① 「x를 같은 크기 옮겼을 때 우리가 찾고자 하는 y를 가장 크게 변동시킨게 가장 큰 핵심인자」 — 변수마다 LNG를 얼마나 크게 움직였나.

**표의 숫자** — [표 1]의 칸은 변수 하나만 바꿔 가며 맞힌 LNG(한 시각 평균)의 가장 큰 값 − 가장 작은 값입니다(MWh). 늘 0 이상이고, 클수록 그 변수가 LNG를 크게 움직였다는 뜻입니다. 햇빛 줄만 낮 10 ~ 15시 시각의 평균입니다. 괄호 안은 그 모델 안에서의 순서입니다. 변수끼리 크기를 비교하는 표이고, 모델끼리 비교하는 표가 아닙니다. [표 2]는 그래디언트 부스팅에서 그 폭이 어디서 나왔는지, 곧 가장 많이 맞힌 값과 가장 적게 맞힌 값을 보입니다.

**넣어 보는 값**

| 변수 | 넣어 보는 값 | 같이 정해 둔 것 |
|---|---|---|
| 기온 | 흔한 범위(배운 두 해 시각의 양 끝 5%씩을 뺀 범위)의 양 끝과 그 사이 1도마다 | 3시간 전 기온도 같은 값으로 넣어, 3시간 동안 바뀐 만큼은 0(1-마와 같다) |
| 햇빛(일사) | 낮 10 ~ 15시 시각만, 흔한 범위를 열여섯 값으로 | 밤에는 늘 0이라 바꿀 수 없어서 낮만 본다 |
| 몇 시인가 | 0 ~ 23시 | 일사는 0으로 — 밤에 해가 뜬 시각이 생기지 않게(1-마와 같다) |
| 몇 월인가 | 1 ~ 12월 | — |
| 무슨 날인가 | 월 · 화~금 · 토 · 일 · 공휴일 · 명절 | 트리 계열에는 화~금 · 공휴일 · 명절을 수요일로 넣는다 |

In [104]:
# 8단계 모델 확정과 핵심인자 · 2-나
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 모델 넷의 설정과 열이 준비 ② · ③에서 만들어집니다"
배울곳 = 전체["year"].isin([2023, 2024])                                        # 1-마와 같은 두 해로 배운다 — 2025년 LNG는 쓰지 않는다
틀 = 전체[배울곳].copy()                                                          # 배운 두 해의 시각 — 여기서 한 가지만 바꿔 다시 맞혀 본다
다시 = {이름: 만들기().fit(입력[열][배울곳], y[배울곳]) for 이름, (만들기, 열) in 모델들.items()}   # 1-마와 같은 설정 · 같은 random_state라 1-마와 같은 모델

def 평균예측(d):                                                                 # d의 시각을 모델 넷으로 맞힌 값의 평균(한 시각 MWh) — 1-마와 같다
    둘 = pd.concat([d, 틀], ignore_index=True)                                    # 원래 시각을 뒤에 붙여야 0/1 열이 배울 때와 같은 열로 만들어진다
    n = len(d)
    열들 = {"선형회귀": 선형열(둘), "기온 × 시간대 선형회귀": 시간대기온열(둘), "트리 계열": 트리열(둘)}
    return {이름: float(다시[이름].predict(열들[열].iloc[:n][입력[열].columns]).mean()) for 이름, (_, 열) in 모델들.items()}

def 기온을(d, g):                                                                # 지금 기온과 3시간 전 기온을 모두 g도로 — 3시간 동안 바뀐 만큼은 0(1-마와 같다)
    d = d.copy()
    d["ta"] = g
    d["3시간 전 기온"] = g
    d["16도보다 추운 만큼"] = d["3시간 전 16도보다 추운 만큼"] = max(16 - g, 0)
    d["24도보다 더운 만큼"] = d["3시간 전 24도보다 더운 만큼"] = max(g - 24, 0)
    return d

def 일사를(d, g):                                                                # 일사만 g로
    d = d.copy()
    d["icsr"] = g
    return d

def 시간을(d, h):                                                                # 모든 시각을 h시로, 일사는 0으로 — 밤에 해가 뜬 시각이 생기지 않게(1-마와 같다)
    d = d.copy()
    d["hour24"] = h
    d["icsr"] = 0.0
    return d

def 달을(d, m):                                                                  # 모든 시각을 m월로
    d = d.copy()
    d["month"] = m
    return d

날값 = {"월": (0, 0, 0), "화~금": (2, 0, 0), "토": (5, 0, 0), "일": (6, 0, 0), "공휴일": (2, 1, 0), "명절": (2, 1, 1)}   # (요일, 공휴일, 명절) — 화~금 · 공휴일 · 명절은 수요일로 넣는다
def 날을(d, 종류):                                                               # 모든 시각을 그 날 종류로
    d = d.copy()
    d["날 종류"] = 종류
    d["dow"], d["is_holiday"], d["명절"] = 날값[종류]
    return d

낮 = 틀["hour24"].between(10, 15)                                                # 낮 10 ~ 15시
기온끝 = 틀["ta"].quantile([0.05, 0.95])                                          # 흔한 범위 — 배운 두 해 시각의 양 끝 5%씩을 뺀다(1-마와 같다)
일사끝 = 틀.loc[낮, "icsr"].quantile([0.05, 0.95])                                # 낮 시각의 일사도 같은 방법으로
훑기 = {                                                                          # 변수: (어느 시각에서, 넣어 볼 값들, 바꾸는 함수, 값을 적는 방법)
    "기온": (틀, np.unique(np.r_[기온끝[0.05], np.arange(np.ceil(기온끝[0.05]), np.floor(기온끝[0.95]) + 1), 기온끝[0.95]]), 기온을, lambda v: f"{v:g}도"),   # 양 끝(1-마와 같은 값)과 그 사이 1도마다
    "햇빛(일사) — 낮 10 ~ 15시만": (틀[낮], np.linspace(일사끝[0.05], 일사끝[0.95], 16), 일사를, lambda v: f"{v:.2f} MJ/m²"),
    "몇 시인가 — 일사 0": (틀, range(24), 시간을, lambda v: f"{v}시"),
    "몇 월인가": (틀, range(1, 13), 달을, lambda v: f"{v}월"),
    "무슨 날인가": (틀, list(날값), 날을, lambda v: v),
}
폭, 끝 = {}, {}
for 변수, (d, 값들, 바꾸기, 적기) in 훑기.items():
    맞힘 = pd.DataFrame({v: 평균예측(바꾸기(d, v)) for v in 값들})               # 줄: 모델, 칸: 넣어 본 값 — 칸마다 맞힌 LNG의 평균
    폭[변수] = 맞힘.max(axis=1) - 맞힘.min(axis=1)                                # 가장 많이 맞힌 값 − 가장 적게 맞힌 값
    g = 맞힘.loc["그래디언트 부스팅"]                                              # [표 2]에 계산을 보일 모델
    끝[변수] = {"가장 많이 맞힌 때": f"{적기(g.idxmax())} · {g.max():,.0f}",
                "가장 적게 맞힌 때": f"{적기(g.idxmin())} · {g.min():,.0f}",
                "폭 = 앞 − 뒤": f"{g.max() - g.min():,.0f}"}

표 = pd.DataFrame(폭).T                                                           # 줄: 변수, 칸: 모델
평균 = 표.mean(axis=1)
순위 = 표.rank(ascending=False).astype(int)                                       # 모델 안에서의 순서
보기 = 표.apply(lambda c: [f"{v:,.0f} ({r})" for v, r in zip(c, 순위[c.name])])   # 「폭 (그 모델 안의 순서)」
보기["넷의 평균"] = [f"{v:,.0f}" for v in 평균]
보기 = 보기.loc[평균.sort_values(ascending=False).index]
보기.insert(0, "순서", range(1, len(보기) + 1))

print("[표 1] 변수마다 LNG를 움직인 폭 (MWh, 한 시각 평균) — 흔히 있는 값을 다 넣어 봤을 때 가장 많이 맞힌 값 − 가장 적게 맞힌 값")
print("        괄호는 그 모델 안의 순서 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 햇빛 줄만 낮 10 ~ 15시 시각의 평균이다")
display(보기.rename_axis("변수"))
print("[표 2] 그래디언트 부스팅에서 [표 1]의 폭이 나온 계산 — 어디서 가장 많이 · 가장 적게 맞혔나 (MWh, 한 시각 평균)")
display(pd.DataFrame(끝).T.loc[보기.index].rename_axis("변수"))

[표 1] 변수마다 LNG를 움직인 폭 (MWh, 한 시각 평균) — 흔히 있는 값을 다 넣어 봤을 때 가장 많이 맞힌 값 − 가장 적게 맞힌 값
        괄호는 그 모델 안의 순서 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 햇빛 줄만 낮 10 ~ 15시 시각의 평균이다


,순서,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅,넷의 평균
변수,,,,,,
몇 시인가 — 일사 0,1,"12,103 (1)","12,688 (1)","9,008 (1)","9,547 (1)","10,837"
햇빛(일사) — 낮 10 ~ 15시만,2,"7,995 (2)","7,476 (2)","6,229 (2)","6,903 (2)","7,150"
무슨 날인가,3,"6,257 (3)","6,154 (3)","5,098 (3)","5,201 (3)","5,677"
기온,4,"5,401 (4)","5,046 (4)","3,832 (4)","4,125 (4)","4,601"
몇 월인가,5,"2,680 (5)","2,636 (5)","1,630 (5)","2,104 (5)","2,262"


[표 2] 그래디언트 부스팅에서 [표 1]의 폭이 나온 계산 — 어디서 가장 많이 · 가장 적게 맞혔나 (MWh, 한 시각 평균)


,가장 많이 맞힌 때,가장 적게 맞힌 때,폭 = 앞 − 뒤
변수,,,
몇 시인가 — 일사 0,"16시 · 22,134","3시 · 12,587","9,547"
햇빛(일사) — 낮 10 ~ 15시만,"0.21 MJ/m² · 22,481","3.28 MJ/m² · 15,578","6,903"
무슨 날인가,"화~금 · 19,464","일 · 14,263","5,201"
기온,"-3.2도 · 19,656","19도 · 15,531","4,125"
몇 월인가,"3월 · 19,007","11월 · 16,903","2,104"


## 2-다 — 변수 하나의 값만 뒤섞으면 일 년 전체에서 얼마나 더 틀리나 (순열 중요도, 모델 넷)

**이 셀이 답하는 물음** — 2-나의 순서가 다른 방법으로 봐도 같은가.

**어떻게 내나.** 2023년으로 배운 모델 넷으로 2024년을 맞히면서, 변수 하나의 값만 시각끼리 뒤섞습니다. 예를 들어 기온을 뒤섞으면 7월 오후 시각에 1월 새벽의 기온이 들어갑니다. 모델이 그 변수를 많이 썼다면 그만큼 많이 틀리게 됩니다. 뒤섞은 뒤 평균 절대 오차 − 원래 평균 절대 오차가 그 변수의 크기이고, 다섯 번 뒤섞어 평균 냅니다. 한 변수에서 나온 열(기온이면 16도보다 추운 만큼 · 3시간 전 기온 같은 것)은 같이 뒤섞습니다. 6단계에서 연습한 것과 같은 「2023년으로 배워 2024년 맞히기」라서 2025년 LNG는 쓰지 않습니다. 돌리는 데 10초쯤 걸립니다.

**표의 숫자** — [표 1]의 칸은 뒤섞어서 늘어난 평균 절대 오차입니다(MWh, 한 시각 평균). 클수록 그 모델이 그 변수를 많이 썼다는 뜻이고, 0이면 그 변수가 없어도 같다는 뜻입니다. 괄호는 그 모델 안에서 몇 번째인지입니다. 변수끼리 비교하는 표입니다. [표 2]는 그래디언트 부스팅에서 이 숫자가 나온 계산, 곧 원래 평균 절대 오차와 뒤섞은 뒤 평균 절대 오차를 보입니다. [표 3]은 이 셀의 물음에 바로 답하는 표입니다. 변수마다 2-나(LNG를 움직인 폭)와 2-다(뒤섞었을 때 늘어난 오차)에서 몇 번째인지를 나란히 놓았습니다.

In [105]:
# 8단계 모델 확정과 핵심인자 · 2-다
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 모델 넷의 설정과 열이 준비 ② · ③에서 만들어집니다"
배운곳 = 전체["year"] == 2023                                                    # 6단계의 연습과 같다 — 2023년으로 배워
맞힌곳 = 전체["year"] == 2024                                                    # 2024년을 맞힌다(2025년 LNG는 쓰지 않는다)
연습 = {이름: 만들기().fit(입력[열][배운곳], y[배운곳]) for 이름, (만들기, 열) in 모델들.items()}   # 준비 ③의 2024년 모델과 같은 설정 · 같은 random_state
틀24 = 전체[맞힌곳].copy()                                                         # 2024년 시각 — 여기서 변수 하나의 값만 시각끼리 뒤섞는다
실24 = y[맞힌곳]                                                                   # 2024년 실제 LNG

def 오차(d):                                                                     # d의 시각을 모델 넷으로 맞혔을 때 평균 절대 오차(한 시각 MWh)
    둘 = pd.concat([d, 전체[배운곳]], ignore_index=True)                           # 2023년 시각을 뒤에 붙여야 0/1 열이 배울 때와 같은 열로 만들어진다(2024년에는 8월이 없다)
    n = len(d)
    열들 = {"선형회귀": 선형열(둘), "기온 × 시간대 선형회귀": 시간대기온열(둘), "트리 계열": 트리열(둘)}
    return {이름: float(np.abs(실24.values - 연습[이름].predict(열들[열].iloc[:n][입력[열].columns])).mean()) for 이름, (_, 열) in 모델들.items()}

묶음 = {                                                                          # 변수 하나 = 같이 뒤섞을 열들(한 변수에서 나온 열은 같은 순서로 뒤섞어야 서로 맞는다)
    "몇 시인가": ["hour24"],
    "햇빛(일사)": ["icsr"],
    "무슨 날인가": ["날 종류", "dow", "is_holiday", "명절"],
    "기온": ["ta", "3시간 전 기온", "16도보다 추운 만큼", "24도보다 더운 만큼", "3시간 전 16도보다 추운 만큼", "3시간 전 24도보다 더운 만큼"],
    "몇 월인가": ["month"],
}
나순서 = {"몇 시인가": 1, "햇빛(일사)": 2, "무슨 날인가": 3, "기온": 4, "몇 월인가": 5}   # 2-나 [표 1]의 순서(넷의 평균)

원래 = 오차(틀24)                                                                  # 뒤섞기 전 평균 절대 오차
늘어남, 섞은뒤 = {}, {}
for 변수, 열 in 묶음.items():
    합 = {m: 0.0 for m in 모델들}
    for 번 in range(5):                                                           # 다섯 번 뒤섞어 평균 낸다 — 뒤섞는 순서에 따라 조금씩 달라서
        d = 틀24.copy()
        섞음 = np.random.default_rng(번).permutation(len(d))                       # 시각의 순서를 무작위로 바꾼 번호
        d[열] = 틀24[열].values[섞음]                                                # 그 변수의 값만 다른 시각의 값으로 바꾼다
        for m, v in 오차(d).items(): 합[m] += v / 5
    섞은뒤[변수] = 합
    늘어남[변수] = {m: 합[m] - 원래[m] for m in 모델들}                               # 뒤섞은 뒤 오차 − 원래 오차

표 = pd.DataFrame(늘어남).T                                                       # 줄: 변수, 칸: 모델
평균 = 표.mean(axis=1)
순위 = 표.rank(ascending=False).astype(int)
보기 = 표.apply(lambda c: [f"{v:,.0f} ({r})" for v, r in zip(c, 순위[c.name])])   # 「늘어난 양 (그 모델 안에서 몇 번째)」
보기["넷의 평균"] = [f"{v:,.0f}" for v in 평균]
보기 = 보기.loc[평균.sort_values(ascending=False).index]
다순서 = {v: i + 1 for i, v in enumerate(보기.index)}                              # 넷의 평균으로 몇 번째인가

print("[표 1] 변수 하나의 값만 시각끼리 뒤섞었을 때 평균 절대 오차가 늘어난 양 (MWh, 한 시각 평균) — 2023년으로 배워 2024년을 맞힘")
print("        괄호는 그 모델 안에서 몇 번째인가 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 다섯 번 뒤섞어 평균 냈다")
display(보기.rename_axis("변수"))
g = "그래디언트 부스팅"
print(f"[표 2] {g}에서 [표 1]이 나온 계산 — 원래 평균 절대 오차 {원래[g]:,.0f} MWh")
display(pd.DataFrame({"뒤섞은 뒤 평균 절대 오차": [f"{섞은뒤[v][g]:,.0f}" for v in 보기.index],
                      "늘어난 양 = 앞 − 원래": [f"{섞은뒤[v][g] - 원래[g]:,.0f}" for v in 보기.index]}, index=보기.index).rename_axis("변수"))
print("[표 3] 2-나와 2-다에서 몇 번째인가 — 둘 다 모델 넷의 평균으로 · 줄은 2-나의 순서대로")
비교 = pd.DataFrame({"2-나에서 몇 번째 (LNG를 움직인 폭)": 나순서, "2-다에서 몇 번째 (뒤섞었을 때 늘어난 오차)": 다순서})
비교["2-나에서 2-다로"] = ["같다" if a == b else ("올라감" if b < a else "내려감") for a, b in zip(비교.iloc[:, 0], 비교.iloc[:, 1])]
display(비교.sort_values(비교.columns[0]).rename_axis("변수"))

[표 1] 변수 하나의 값만 시각끼리 뒤섞었을 때 평균 절대 오차가 늘어난 양 (MWh, 한 시각 평균) — 2023년으로 배워 2024년을 맞힘
        괄호는 그 모델 안에서 몇 번째인가 · 「넷의 평균」이 큰 순서로 줄을 놓았다 · 다섯 번 뒤섞어 평균 냈다


,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅,넷의 평균
변수,,,,,
몇 시인가,"3,343 (1)","3,589 (1)","2,114 (1)","2,648 (1)","2,923"
무슨 날인가,"1,422 (2)","1,430 (2)","1,390 (2)","1,431 (2)","1,418"
기온,"1,103 (4)","1,180 (4)",786 (3),940 (3),"1,002"
햇빛(일사),"1,329 (3)","1,218 (3)",453 (4),582 (4),896
몇 월인가,214 (5),225 (5),359 (5),323 (5),280


[표 2] 그래디언트 부스팅에서 [표 1]이 나온 계산 — 원래 평균 절대 오차 2,004 MWh


,뒤섞은 뒤 평균 절대 오차,늘어난 양 = 앞 − 원래
변수,,
몇 시인가,"4,651","2,648"
무슨 날인가,"3,434","1,431"
기온,"2,944",940
햇빛(일사),"2,585",582
몇 월인가,"2,326",323


[표 3] 2-나와 2-다에서 몇 번째인가 — 둘 다 모델 넷의 평균으로 · 줄은 2-나의 순서대로


,2-나에서 몇 번째 (LNG를 움직인 폭),2-다에서 몇 번째 (뒤섞었을 때 늘어난 오차),2-나에서 2-다로
변수,,,
몇 시인가,1,1,같다
햇빛(일사),2,4,내려감
무슨 날인가,3,2,올라감
기온,4,3,올라감
몇 월인가,5,5,같다


## 2-라 — 실제 LNG를 변수의 값마다 묶어 평균 내면, 변수마다 LNG가 얼마나 달랐나 (모델 없이 · 탐색적 기법 결과)

**이 셀이 답하는 물음** — 2023 · 2024년에 실제로 기록된 LNG를 변수의 값마다 묶어 평균을 내면, 평균이 가장 많은 묶음과 가장 적은 묶음의 차이(폭)가 얼마인가. 다섯 변수를 그 폭이 큰 순서로 세우면, 2-나(모델이 맞힌 값으로 본 폭)에서 세운 순서와 같은가.

**어떻게 내나.** 모델이 맞힌 값은 쓰지 않습니다. 「시각」은 한 시간짜리 기록 하나입니다(2023년 1월 1일 18시가 시각 하나). 두 해의 시각을 그 변수의 값이 같은 것끼리 묶습니다. 몇 시인가라면 0시인 시각끼리, 1시인 시각끼리, … 23시인 시각끼리 스물네 묶음입니다. 묶음마다 실제 LNG를 평균 내고, 그 평균 가운데 가장 많은 것에서 가장 적은 것을 뺀 값이 그 변수의 폭입니다. 이렇게 묶으면 묶음끼리 날씨 같은 다른 조건을 같게 맞출 수 없습니다. 그래서 예를 들어 1월 묶음의 평균에는 추운 날씨 때문에 늘어난 LNG가 섞여 있습니다.

**표의 숫자**

- 모두 한 시각의 LNG 발전량(MWh)입니다. 폭은 그런 평균 둘을 뺀 값입니다.
- 햇빛 줄만 낮 10 ~ 15시(10 · 11 · 12 · 13 · 14 · 15시, 여섯 시간)의 시각으로 냅니다. 밤에는 햇빛이 늘 0이라서 2-나와 같게 낮만 봅니다.
- 기온과 햇빛은 흔한 범위 안의 시각만 씁니다. 흔한 범위는 값을 작은 것부터 줄 세웠을 때 양 끝 5%씩을 뺀 범위입니다(2-나와 같습니다. 햇빛은 낮 시각으로 정합니다). 범위의 두 끝은 [표 2] 「어떻게 묶었나」 칸에 있습니다.
- [표 1] — 두 폭과, 그 폭으로 다섯 변수를 세운 순서입니다. 「모델이 맞힌 값으로 본 폭」은 2-나 [표 1]의 「넷의 평균」 칸을 옮겨 적은 것입니다. 9단계에 가는 모델은 그래디언트 부스팅 하나지만, 핵심인자는 모델 하나로 정하지 않기로 해서(2-가) 순서를 모델 넷의 평균으로 세웁니다.
- [표 2] — [표 1]의 「실제 LNG로 본 폭」이 나온 계산입니다. 변수마다 어떻게 묶었는지, 평균이 가장 많은 묶음과 가장 적은 묶음, 그 뺄셈을 보입니다.
- [표 3] — 그 계산을 몇 월인가 하나로 열두 묶음 모두 보인 것입니다. 옆에 달만 그 달로 바꿨을 때 모델이 맞힌 평균을 놓았습니다. 이 칸은 계산을 보이려고 모델 넷 가운데 그래디언트 부스팅(물음 1에서 고른 모델) 하나로 냈습니다. 그래서 [표 1]의 모델 칸(모델 넷의 평균)과 숫자가 다릅니다.

In [106]:
# 8단계 모델 확정과 핵심인자 · 2-라
assert "두번" in globals(), "준비 ③을 먼저 돌려 주세요 — 준비 ② · ③에서 표와 열이 만들어집니다"
배운곳 = 전체["year"].isin([2023, 2024])                                            # 배운 두 해의 시각 — 2025년 LNG는 쓰지 않는다
d = 전체[배운곳].copy()
d["LNG"] = y[d.index]                                                             # 그 시각에 실제로 기록된 LNG(MWh) — 모델이 맞힌 값이 아니다
기온끝 = d["ta"].quantile([0.05, 0.95])                                            # 흔한 범위 — 작은 것부터 줄 세워 양 끝 5%씩을 뺀다(2-나와 같다)
낮 = d["hour24"].between(10, 15)                                                  # 낮 10 ~ 15시(10 · 11 · 12 · 13 · 14 · 15시)
일사끝 = d.loc[낮, "icsr"].quantile([0.05, 0.95])                                  # 햇빛의 흔한 범위는 낮 시각으로 정한다
온d = d[d["ta"].between(기온끝[0.05], 기온끝[0.95])]                               # 흔한 범위 안의 기온인 시각
낮d = d[낮 & d["icsr"].between(일사끝[0.05], 일사끝[0.95])]                         # 흔한 범위 안의 햇빛인 낮 시각

def 묶어서(시각들, 값):                                                             # 값이 같은 시각끼리 묶어, 묶음마다 시각 수와 실제 LNG의 평균을 낸다
    g = 시각들.groupby(값, observed=True)["LNG"]
    return pd.DataFrame({"시각 수": g.size(), "평균": g.mean().round()})          # 평균을 반올림해 두어야 아래 뺄셈이 표에 적힌 숫자로 맞는다

묶음들 = {                                                                         # 변수: (어떻게 묶었나, 묶음마다 시각 수와 평균)
    "몇 시인가": ("0 ~ 23시마다", 묶어서(d, d["hour24"]).rename(index=lambda v: f"{v}시")),
    "햇빛(일사) — 낮 10 ~ 15시만": (f"낮 시각의 흔한 범위({일사끝[0.05]:.2f} ~ {일사끝[0.95]:.2f} MJ/m²)를 같은 너비의 열다섯 구간으로",
                                   묶어서(낮d, pd.cut(낮d["icsr"], np.linspace(일사끝[0.05], 일사끝[0.95], 16), include_lowest=True))
                                   .rename(index=lambda v: f"{v.left:.2f} ~ {v.right:.2f} MJ/m²")),
    "무슨 날인가": ("날 종류(월 · 화~금 · 토 · 일 · 공휴일 · 명절)마다", 묶어서(d, d["날 종류"])),
    "기온": (f"흔한 범위({기온끝[0.05]:.1f} ~ {기온끝[0.95]:.1f}도) 안에서 반올림해 1도마다", 묶어서(온d, 온d["ta"].round()).rename(index=lambda v: f"{v:.0f}도")),
    "몇 월인가": ("1 ~ 12월마다", 묶어서(d, d["month"]).rename(index=lambda v: f"{v}월")),
}
많은 = {v: m["평균"].idxmax() for v, (_, m) in 묶음들.items()}                      # 실제 LNG 평균이 가장 많은 묶음
적은 = {v: m["평균"].idxmin() for v, (_, m) in 묶음들.items()}                      # 실제 LNG 평균이 가장 적은 묶음
폭 = pd.Series({v: m.loc[많은[v], "평균"] - m.loc[적은[v], "평균"] for v, (_, m) in 묶음들.items()})
나폭 = pd.Series({"몇 시인가": 10837, "햇빛(일사) — 낮 10 ~ 15시만": 7150, "무슨 날인가": 5677, "기온": 4601, "몇 월인가": 2262})   # 2-나 [표 1] 「넷의 평균」(10-12에 돌린 값)
줄 = 나폭.sort_values(ascending=False).index                                       # 줄은 2-나의 순서대로

print("[표 1] 변수마다 LNG가 얼마나 달랐나 (한 시각의 LNG, MWh) — 실제 LNG로 본 폭(모델 없이)과 모델이 맞힌 값으로 본 폭(2-나)")
display(pd.DataFrame({"실제 LNG로 본 폭 (모델 없이)": 폭.map("{:,.0f}".format),
                      "모델이 맞힌 값으로 본 폭 (2-나 · 모델 넷이 각각 낸 폭의 평균)": 나폭.map("{:,.0f}".format),
                      "실제 LNG로 몇 번째 (1이 가장 큼)": 폭.rank(ascending=False).astype(int),
                      "모델로 몇 번째 (2-나 · 1이 가장 큼)": 나폭.rank(ascending=False).astype(int)}).loc[줄].rename_axis("변수"))

print("[표 2] [표 1]의 「실제 LNG로 본 폭」이 나온 계산 — 두 해의 시각을 그 변수의 값이 같은 것끼리 묶고, 묶음마다 실제 LNG를 평균 내서, 평균이 가장 많은 묶음 − 가장 적은 묶음")
display(pd.DataFrame({"어떻게 묶었나": {v: f"{방법} — {len(m)}묶음" for v, (방법, m) in 묶음들.items()},
                      "실제 LNG 평균이 가장 많은 묶음 (그 묶음의 시각 수)": {v: f"{많은[v]} (시각 {m.loc[많은[v], '시각 수']:,}개)" for v, (_, m) in 묶음들.items()},
                      "실제 LNG 평균이 가장 적은 묶음 (그 묶음의 시각 수)": {v: f"{적은[v]} (시각 {m.loc[적은[v], '시각 수']:,}개)" for v, (_, m) in 묶음들.items()},
                      "폭 = 가장 많은 묶음의 평균 − 가장 적은 묶음의 평균": {v: f"{m.loc[많은[v], '평균']:,.0f} − {m.loc[적은[v], '평균']:,.0f} = {폭[v]:,.0f}"
                                                                 for v, (_, m) in 묶음들.items()}}).loc[줄].rename_axis("변수"))

gb = 모델들["그래디언트 부스팅"][0]().fit(입력["트리 계열"][배운곳], y[배운곳])      # 2-나와 같은 두 해 · 같은 설정으로 배운다 — 2-나의 그래디언트 부스팅과 같은 모델
x = 입력["트리 계열"][배운곳].copy()                                              # 두 해의 모든 시각
gb월 = {}
for 달 in range(1, 13):
    x["month"] = 달                                                               # 모든 시각을 그 달로 — 기온 · 햇빛 · 몇 시 · 무슨 날은 시각마다 그대로
    gb월[f"{달}월"] = round(float(gb.predict(x).mean()))                          # 모델이 맞힌 LNG의 평균
gb월 = pd.Series(gb월)
월 = 묶음들["몇 월인가"][1]
print("[표 3] [표 2]의 계산을 몇 월인가로 열두 묶음 모두 — 달마다 LNG 평균 (한 시각의 LNG, MWh · 2023 · 2024년)")
print("        「모델이 맞힌 평균」은 두 해의 모든 시각에서 달만 그 달로 바꿔 넣고 그래디언트 부스팅(물음 1에서 고른 모델)이 맞힌 LNG의 평균이다.")
print("        예를 들어 7월 14시 시각이면 기온 · 햇빛 · 몇 시 · 무슨 날은 그 시각의 값 그대로 두고 달만 1월로 넣는다.")
print("        열두 달 모두 같은 날씨 · 같은 시각으로 맞히니, 달끼리의 차이는 달 하나만 바꾼 차이다.")
display(pd.DataFrame({"그 달의 시각 수": 월["시각 수"].map("{:,}".format),
                      "그 달의 평균 기온 (도)": d.groupby("month")["ta"].mean().rename(index=lambda v: f"{v}월").map("{:.1f}".format),
                      "실제 LNG의 평균 (모델 없이)": 월["평균"].map("{:,.0f}".format),
                      "모델이 맞힌 평균 (그래디언트 부스팅 · 모든 시각을 그 달로)": gb월.map("{:,}".format)}).rename_axis("달"))
print(f"실제 LNG — 가장 많은 {많은['몇 월인가']} {월['평균'].max():,.0f} − 가장 적은 {적은['몇 월인가']} {월['평균'].min():,.0f} = {폭['몇 월인가']:,.0f}"
      f" ([표 1]의 몇 월인가 「실제 LNG로 본 폭」과 같다)")
print(f"모델이 맞힌 값 — 가장 많은 {gb월.idxmax()} {gb월.max():,} − 가장 적은 {gb월.idxmin()} {gb월.min():,} = {gb월.max() - gb월.min():,}"
      f" (2-나에서 그래디언트 부스팅이 낸 몇 월인가 폭과 같다. [표 1]의 {나폭['몇 월인가']:,}은 모델 넷이 낸 폭의 평균이라 다르다)")

[표 1] 변수마다 LNG가 얼마나 달랐나 (한 시각의 LNG, MWh) — 실제 LNG로 본 폭(모델 없이)과 모델이 맞힌 값으로 본 폭(2-나)


,실제 LNG로 본 폭 (모델 없이),모델이 맞힌 값으로 본 폭 (2-나 · 모델 넷이 각각 낸 폭의 평균),실제 LNG로 몇 번째 (1이 가장 큼),모델로 몇 번째 (2-나 · 1이 가장 큼)
변수,,,,
몇 시인가,"9,782","10,837",1,1
햇빛(일사) — 낮 10 ~ 15시만,"6,034","7,150",3,2
무슨 날인가,"5,326","5,677",5,3
기온,"7,213","4,601",2,4
몇 월인가,"5,423","2,262",4,5


[표 2] [표 1]의 「실제 LNG로 본 폭」이 나온 계산 — 두 해의 시각을 그 변수의 값이 같은 것끼리 묶고, 묶음마다 실제 LNG를 평균 내서, 평균이 가장 많은 묶음 − 가장 적은 묶음


,어떻게 묶었나,실제 LNG 평균이 가장 많은 묶음 (그 묶음의 시각 수),실제 LNG 평균이 가장 적은 묶음 (그 묶음의 시각 수),폭 = 가장 많은 묶음의 평균 − 가장 적은 묶음의 평균
변수,,,,
몇 시인가,0 ~ 23시마다 — 24묶음,18시 (시각 699개),3시 (시각 699개),"22,396 − 12,614 = 9,782"
햇빛(일사) — 낮 10 ~ 15시만,낮 시각의 흔한 범위(0.21 ~ 3.28 MJ/m²)를 같은 너비의 열다섯 구간으...,0.21 ~ 0.41 MJ/m² (시각 281개),3.08 ~ 3.28 MJ/m² (시각 184개),"22,427 − 16,393 = 6,034"
무슨 날인가,날 종류(월 · 화~금 · 토 · 일 · 공휴일 · 명절)마다 — 6묶음,"화~금 (시각 9,096개)","일 (시각 2,370개)","19,446 − 14,120 = 5,326"
기온,흔한 범위(-3.2 ~ 29.1도) 안에서 반올림해 1도마다 — 33묶음,29도 (시각 212개),14도 (시각 512개),"22,144 − 14,931 = 7,213"
몇 월인가,1 ~ 12월마다 — 12묶음,"1월 (시각 1,485개)","10월 (시각 1,488개)","20,648 − 15,225 = 5,423"


[표 3] [표 2]의 계산을 몇 월인가로 열두 묶음 모두 — 달마다 LNG 평균 (한 시각의 LNG, MWh · 2023 · 2024년)
        「모델이 맞힌 평균」은 두 해의 모든 시각에서 달만 그 달로 바꿔 넣고 그래디언트 부스팅(물음 1에서 고른 모델)이 맞힌 LNG의 평균이다.
        예를 들어 7월 14시 시각이면 기온 · 햇빛 · 몇 시 · 무슨 날은 그 시각의 값 그대로 두고 달만 1월로 넣는다.
        열두 달 모두 같은 날씨 · 같은 시각으로 맞히니, 달끼리의 차이는 달 하나만 바꾼 차이다.


,그 달의 시각 수,그 달의 평균 기온 (도),실제 LNG의 평균 (모델 없이),모델이 맞힌 평균 (그래디언트 부스팅 · 모든 시각을 그 달로)
달,,,,
1월,"1,485",-1.0,"20,648","17,672"
2월,"1,344",3.0,"19,035","17,570"
3월,"1,485",8.4,"18,933","19,007"
4월,"1,440",15.0,"16,115","17,598"
5월,"1,488",19.0,"15,726","17,456"
6월,"1,440",24.0,"16,956","17,504"
7월,"1,488",26.6,"19,203","17,752"
8월,744,27.2,"19,229","17,407"
9월,"1,437",24.6,"18,090","17,627"


실제 LNG — 가장 많은 1월 20,648 − 가장 적은 10월 15,225 = 5,423 ([표 1]의 몇 월인가 「실제 LNG로 본 폭」과 같다)
모델이 맞힌 값 — 가장 많은 3월 19,007 − 가장 적은 11월 16,903 = 2,104 (2-나에서 그래디언트 부스팅이 낸 몇 월인가 폭과 같다. [표 1]의 2,262은 모델 넷이 낸 폭의 평균이라 다르다)


## 2-마 — 다섯 변수의 순서를 세 방법으로 나란히 놓고, 결론에 어떻게 적을지 정하기

**이 셀이 답하는 물음** — 다섯 변수가 LNG를 움직인 순서를 결론 문장에 어떻게 적을까.

**표의 숫자** — 그 방법에서 몇 번째였는지입니다. 1이 LNG를 가장 크게 움직인 변수입니다.

| 변수 | 2-나 — 그 변수만 바꿔 넣었을 때 LNG가 움직인 폭 (2-가에서 정한 기준) | 2-다 — 그 변수만 시각끼리 뒤섞었을 때 더 틀린 양 | 2-라 — 모델 없이 실제 기록을 묶어 본 폭 |
|---|---|---|---|
| 몇 시인가 | 1 | 1 | 1 |
| 햇빛(일사) | 2 — 낮 10 ~ 15시만 봤다 | 4 (선형회귀 둘은 3) — 일 년의 모든 시각을 봤다 | 3 — 낮 10 ~ 15시만 봤다 |
| 무슨 날인가 | 3 | 2 | 5 |
| 기온 | 4 | 3 | 2 |
| 몇 월인가 | 5 | 5 | 4 |

- 2-나 · 2-다는 모델 넷의 평균으로 매긴 순서입니다.
- 햇빛을 빼고 보면, 모델로 본 두 방법(2-나 · 2-다)은 몇 시인가 → 무슨 날인가 → 기온 → 몇 월인가로 같습니다. 두 방법 사이에 달라진 것은 햇빛이 몇 번째인지 하나입니다.
- 2-나에서 햇빛만 낮 시각으로 본 것은, 밤에는 햇빛이 늘 0이라 밤 시각에 햇빛 값을 넣어 보면, 새벽 2시에 한낮의 햇빛 값이 붙은 것처럼 실제로는 없는 조합이 생기기 때문입니다.

## 2-바 — 모델에 넣은 다섯의 여덟 칸과, 무엇을 핵심인자라고 부를까

**이 셀이 답하는 물음** — 모델에 넣은 다섯 변수가 각각 어떤 길로 LNG를 움직인다고 봤고, 어떻게 넣었고, 얼마나 움직였고, 그 숫자를 믿을 수 있나. 그리고 그 가운데 무엇을 핵심인자라고 부를까 — 이것은 아래 ✍️ 칸에 읽으신 것을 적고 제가 답을 단 뒤, 「✍️ 읽고 나서」에서 여쭙습니다.

**표 읽는 법** — 가로줄 하나가 변수마다 적는 여덟 항목(2-가의 「여덟 칸」) 가운데 하나이고, 세로줄 하나가 변수 하나입니다. 「제외 사유」는 넣지 않은 것에 적는 항목이라 이 표에는 없고, 「모델별 중요도」는 방법이 둘이라 두 줄로 나눴습니다. 변수는 결론에 적기로 한 순서(2-마)대로 놓았습니다.

| 항목 | 몇 시인가 | 무슨 날인가 | 기온 | 햇빛(일사) | 몇 월인가 |
|---|---|---|---|---|---|
| 역할 — 어떤 길로 LNG를 움직인다고 봤나(7단계 인과지도) | 하루의 생활 · 일의 흐름으로 전력 수요가 오르내린다. 낮에는 일사가 늘어 태양광이 LNG를 줄인다 | 쉬는 날 · 연휴에는 사회 활동이 줄어 전력 수요가 준다 | 16도보다 추우면 난방, 24도보다 더우면 냉방으로 전력 수요가 는다. 추우면 열병합(난방 열을 같이 만드는 발전소)도 더 돈다 | 일사가 강할수록 전력시장의 태양광이 더 돌아 LNG가 준다 | 달마다 기온 · 일사 · 원전 정비 · 석탄 상한이 다르다. 기온 · 일사는 따로 넣어서, 몇 월인가에는 그 밖에 달마다 다른 것이 남는다 |
| 형태 — 모델에 어떤 숫자로 넣었나(6단계 4-가) | 선형회귀 0/1 열 23개 · 트리 계열 0 ~ 23 그대로 | 선형회귀 0/1 열 5개(화~금 · 토 · 일 · 공휴일 · 명절, 월요일이 기준)와 쉬는 날이면서 몇 시인가 23개(쉬는 날과 몇 시인가를 곱한 열 — 2-나 · 2-다에서 무슨 날인가나 몇 시인가를 바꾸면 같이 바뀌었다) · 트리 계열 요일 · 공휴일 · 명절 | 선형회귀 16도보다 추운 만큼 · 24도보다 더운 만큼 · 3시간 동안 바뀐 만큼(추운 쪽 · 더운 쪽) · 트리 계열 기온 그대로 | 일사(MJ/m²) 한 열 — 모든 모델 | 선형회귀 0/1 열 11개(1월 기준) · 트리 계열 1 ~ 12 그대로 |
| 탐색적 기법 결과 — 모델을 쓰지 않고 본 폭(2-라): 실제로 기록된 LNG를 그 변수의 값이 같은 시각끼리 묶어 묶음마다 평균을 내고, 평균이 가장 많은 묶음의 평균 − 가장 적은 묶음의 평균 | 1번째 · 9,782 | 5번째 · 5,326 | 2번째 · 7,213 | 3번째 · 6,034 | 4번째 · 5,423 |
| 모델별 중요도 ① 바꿔 넣어 보기(2-나) — 그 변수에 흔히 있는 값(양 끝 5%씩을 뺀 범위)을 하나씩 모든 시각에 넣어 모델로 맞혀 보고, 값마다 낸 평균 가운데 가장 많은 것 − 가장 적은 것 | 1번째 · 10,837 | 3번째 · 5,677 | 4번째 · 4,601 | 2번째 · 7,150 | 5번째 · 2,262 |
| 모델별 중요도 ② 뒤섞어 보기(2-다) — 그 변수의 값만 시각끼리 섞어 놓고 모델로 맞혔을 때, 평균 절대 오차가 섞기 전보다 늘어난 양 | 1번째 · 2,923 | 2번째 · 1,418 | 3번째 · 1,002 | 4번째 · 896 | 5번째 · 280 |
| 기술적 검토 — 숫자를 믿을 수 있나(7단계 1-나 · 1-사) | VIF 10 아래 | VIF 10 아래 · 일요일 기울기의 p값 0.0000 | VIF 10 아래 · 기온 열 넷 가운데 셋은 p값 0.05 아래(가장 큰 것 0.0018), 「3시간 동안 바뀐 만큼(더운 쪽)」 하나만 0.89 | VIF 4.1 · 기울기의 p값 0.0000 | VIF 10 아래 |
| 총점 — 결론에 적는 순서(2-마). 위 줄의 순위를 더한 점수가 아니다 | 1 | 2 | 3 | 4 | 5 |
| 선정 사유 | 핵심인자 1번째 — 뒤섞으면 틀린 양이 가장 많이 늘었다(2,923). 다음인 무슨 날인가의 두 배쯤 | 핵심인자 2번째 — 두 번째로 많이 늘었다(1,418) | 핵심인자 3번째 — 1,002. 햇빛과 크기가 비슷해서 선형회귀 둘에서는 4번째 | 핵심인자 4번째 — 896. 해가 있는 시각에만 LNG를 움직인다(낮 10 ~ 15시만 보면 2번째) | 핵심인자 5번째 — 가장 적게 늘었다(280, 몇 시인가의 10분의 1쯤). 날씨를 그대로 두고 달만 바꾸면 크게 움직이지 않는다(2-라 [표 3]) |

- 숫자의 단위는 한 시각의 MWh이고, 2-나 · 2-다는 모델 넷의 평균입니다. 햇빛만 2-라 · 2-나에서 낮 10 ~ 15시 시각으로 구했고, 나머지는 밤낮의 모든 시각으로 구했습니다. 2-다에서 선형회귀 둘은 햇빛이 3번째 · 기온이 4번째였습니다.
- VIF는 모델 넷 가운데 선형회귀 둘에서 냈습니다. 선형회귀 67열과 기온 × 시간대 선형회귀 113열(기온 두 열을 시간대마다 따로 주어 46열이 더 있다)이 모두 10 아래였고, 가장 큰 것은 4.7 · 5.9였습니다. 표의 VIF 4.1과 p값은 선형회귀에서 냈습니다. p값은 이웃한 시각끼리 같은 쪽으로 틀리는 것(7단계에서 독립성이 깨진 것)을 4주(672시간) 떨어진 시각까지 감안해 다시 낸 값이고, p값으로 변수를 고르지는 않습니다(7단계 물음 1). 트리 계열은 기울기가 없어 p값이 없습니다.
- 「3시간 동안 바뀐 만큼(더운 쪽)」은 기울기가 0과 구별되지 않았지만 그대로 둡니다 — 고른 그래디언트 부스팅은 이 열을 쓰지 않고, 8단계에서는 모델을 고치지 않습니다(클로드 판단). 한계에 적습니다.

## 2-사 — 기온 · 햇빛이 한 칸 움직이면 LNG가 얼마나 움직이나 (1도 · 1 MJ/m²에 몇 MWh, 모델 넷)

**이 셀이 답하는 물음** — 「16도보다 1도 추울 때 LNG가 얼마나 느나」(6단계 물음 7)와, 창주 님 「그러면 그 수치가 있으면 뭐지 스케일링 비교도 되는건 아니야??」(2-바 「✍️ 다시 읽고 나서」 5).

**물음 2의 한 줄 어디에 답하나** — 「디스케일링해가지고 어느 정도의 비중을 변수마다 갖고 있는지 보고」의 디스케일링입니다. 「1도에 얼마」는 디스케일링하기 전의 숫자이고, 여기에 흔히 움직이는 폭을 곱하면 비중을 비교할 때 쓴 크기가 되는지를 봅니다.

**어떻게 내나.** 1-마 [표 2]와 같은 계산입니다. 배운 두 해(2023 · 2024년)의 시각을 그대로 두고 기온이나 햇빛 하나만 바꿔 모델 넷으로 다시 맞힌 뒤, 맞힌 LNG의 평균이 움직인 양을 바꾼 크기로 나눕니다. 바꾸는 폭은 1-마와 같습니다 — 기온 추운 쪽은 16도에서 흔한 범위의 추운 쪽 끝(영하 3.2도)까지, 더운 쪽은 24도에서 더운 쪽 끝(29.1도)까지, 햇빛은 낮 10 ~ 15시에 일사가 약한 끝(0.21)에서 강한 끝(3.28 MJ/m²)까지입니다. 몇 시인가 · 무슨 날인가 · 몇 월인가는 한 칸이 크기를 갖지 않는 범주라 넣지 않았습니다 — 1-마 [표 2]의 「3시 → 18시」 · 「수요일 → 일요일」이 그 대신입니다. 2-나에서 만든 모델과 바꾸는 함수를 쓰니, 준비 셀 셋 → 2-나 → 2-사 순서로 돌립니다. [표 2]의 바꿔 넣어 보기 폭은 2-나와 같은 계산을 이 셀에서 다시 합니다. 2025년 LNG는 쓰지 않습니다.

**표의 숫자** — [표 1]의 칸은 한 칸(1도 · 1 MJ/m²) 움직일 때 모델이 맞힌 LNG가 움직인 양입니다(MWh, 한 시각 평균). +면 LNG가 늘고, −면 줍니다. 줄마다 한 칸의 단위가 달라서 줄끼리 크기를 비교하는 표가 아닙니다. [표 2]는 [표 1]의 넷의 평균에 흔히 움직이는 폭을 곱한 값을, 바꿔 넣어 보기(2-나)에서 비중을 비교할 때 쓴 폭과 나란히 놓았습니다. 곱한 값은 + · −를 떼고 크기만 적었습니다 — 바꿔 넣어 보기의 폭은 「가장 많이 맞힌 값 − 가장 적게 맞힌 값」이라 늘 +이기 때문입니다.

In [107]:
# 8단계 모델 확정과 핵심인자 · 2-사
assert "평균예측" in globals() and "틀" in globals(), "2-나를 먼저 돌려 주세요 — 모델 넷과 「한 가지만 바꿔 맞히기」가 2-나에서 만들어집니다"
낮시각 = 틀["hour24"].between(10, 15)                                             # 낮 10 ~ 15시 — 2-라가 「낮」 · 「폭」 같은 이름을 다시 써서, 이 셀에서 쓸 것은 다른 이름으로 다시 만든다
기온범위 = 틀["ta"].quantile([0.05, 0.95])                                         # 흔한 범위 — 배운 두 해 시각의 양 끝 5%씩을 뺀다(1-마 · 2-나와 같다)
일사범위 = 틀.loc[낮시각, "icsr"].quantile([0.05, 0.95])                           # 낮 시각의 일사도 같은 방법으로
한칸 = {                                                                          # 줄 이름: (바꾸기 전, 바꾼 뒤, 바꾼 크기, 한 칸의 단위) — 1-마 [표 2]와 같은 바꾸기
    "기온 — 16도보다 1도 추울 때마다": (기온을(틀, 16), 기온을(틀, 기온범위[0.05]), 16 - 기온범위[0.05], "도"),
    "기온 — 24도보다 1도 더울 때마다": (기온을(틀, 24), 기온을(틀, 기온범위[0.95]), 기온범위[0.95] - 24, "도"),
    "햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다": (일사를(틀[낮시각], 일사범위[0.05]), 일사를(틀[낮시각], 일사범위[0.95]), 일사범위[0.95] - 일사범위[0.05], "MJ/m²"),
}
한칸에 = {}
for 이름, (앞d, 뒤d, 칸, _) in 한칸.items():
    앞, 뒤 = 평균예측(앞d), 평균예측(뒤d)                                           # 바꾸기 전 · 바꾼 뒤 맞힌 LNG의 평균 — 1-마와 같은 계산
    한칸에[이름] = pd.Series({m: (뒤[m] - 앞[m]) / 칸 for m in 모델들})             # 움직인 양을 바꾼 크기로 나눈다 = 한 칸에 몇 MWh
표1 = pd.DataFrame(한칸에).T                                                      # 줄: 한 칸, 칸: 모델
표1["넷의 평균"] = 표1.mean(axis=1)                                                # 2-가 — 한 모델로 정하지 않는다

print("[표 1] 한 칸 움직일 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — +면 LNG가 는다 · −면 준다")
print("        1-마 [표 2]의 움직인 양을 바꾼 크기로 나눈 값 · 줄마다 한 칸의 단위(도 · MJ/m²)가 달라 줄끼리 비교하는 표가 아니다")
display(표1.apply(lambda c: c.map(lambda v: f"{v:+,.0f}")).rename_axis("한 칸"))

훑을값 = {"기온 — 16도보다 1도 추울 때마다": (틀, np.unique(np.r_[기온범위[0.05], np.arange(np.ceil(기온범위[0.05]), np.floor(기온범위[0.95]) + 1), 기온범위[0.95]]), 기온을),
          "햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다": (틀[낮시각], np.linspace(일사범위[0.05], 일사범위[0.95], 16), 일사를)}   # 2-나와 같은 값들(기온은 1도마다, 햇빛은 열여섯 값)
나폭 = {}
for 줄, (d, 값들, 바꾸기) in 훑을값.items():
    맞힘 = pd.DataFrame({v: 평균예측(바꾸기(d, v)) for v in 값들})               # 줄: 모델, 칸: 넣어 본 값 — 2-나와 같은 계산을 이 셀에서 다시 한다
    나폭[줄] = (맞힘.max(axis=1) - 맞힘.min(axis=1)).mean()                     # 가장 많이 맞힌 값 − 가장 적게 맞힌 값, 모델 넷의 평균 = 2-나 [표 1] 「넷의 평균」
표2 = pd.DataFrame({줄: {"한 칸에 얼마 (넷의 평균)": f"{표1.loc[줄, '넷의 평균']:+,.0f}",
                        "흔히 움직이는 폭": f"{한칸[줄][2]:.1f}도" if 한칸[줄][3] == "도" else f"{한칸[줄][2]:.2f} MJ/m²",
                        "곱한 값 (+ · −를 뗀 크기)": f"{abs(표1.loc[줄, '넷의 평균'] * 한칸[줄][2]):,.0f}",   # 한 칸에 얼마 × 흔히 움직이는 폭
                        "바꿔 넣어 보기(2-나)의 폭 (넷의 평균)": f"{나폭[줄]:,.0f}"}                       # 비중을 비교할 때 쓴 폭
                    for 줄 in 훑을값}).T
print("[표 2] 한 칸에 얼마 × 흔히 움직이는 폭 = 곱한 값 — 바꿔 넣어 보기(2-나)의 폭과 나란히 (MWh, 한 시각 평균)")
display(표2.rename_axis("한 칸"))

[표 1] 한 칸 움직일 때 모델이 맞힌 LNG가 움직인 양 (MWh, 한 시각 평균) — +면 LNG가 는다 · −면 준다
        1-마 [표 2]의 움직인 양을 바꾼 크기로 나눈 값 · 줄마다 한 칸의 단위(도 · MJ/m²)가 달라 줄끼리 비교하는 표가 아니다


,선형회귀,기온 × 시간대 선형회귀,랜덤포레스트,그래디언트 부스팅,넷의 평균
한 칸,,,,,
기온 — 16도보다 1도 추울 때마다,+281,+263,+197,+214,+239
기온 — 24도보다 1도 더울 때마다,+977,+587,+545,+534,+661
햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다,"-2,604","-2,435","-2,029","-2,248","-2,329"


[표 2] 한 칸에 얼마 × 흔히 움직이는 폭 = 곱한 값 — 바꿔 넣어 보기(2-나)의 폭과 나란히 (MWh, 한 시각 평균)


,한 칸에 얼마 (넷의 평균),흔히 움직이는 폭,곱한 값 (+ · −를 뗀 크기),바꿔 넣어 보기(2-나)의 폭 (넷의 평균)
한 칸,,,,
기온 — 16도보다 1도 추울 때마다,+239,19.2도,"4,584","4,601"
햇빛 — 낮에 일사가 1 MJ/m² 강할 때마다,"-2,329",3.07 MJ/m²,"7,150","7,150"


# 물음 3 — 1단계에 짐작으로 적은 원인 여섯 줄이 맞았나

<div style="border: 2px solid var(--vscode-focusBorder, #3794ff); border-left-width: 6px; border-radius: 8px; padding: 4px 18px 10px; background: var(--vscode-textBlockQuote-background, rgba(127, 127, 127, 0.1));">

## 📌 물음 3 결론

**왜 생긴 물음인가** — 1단계(무엇에 답할지 정하는 단계, 09-18)에서 LNG 발전량을 움직일 것 같은 원인을 여섯 줄로 짐작해 적고, 8단계에서 모델이 고른 변수와 맞춰 보기로 했습니다. 물음 2에서 핵심인자 다섯을 정했으니, 여섯 줄이 맞았는지 하나씩 맞춰 본 점검입니다. 창주 님 말로는 「원인으로 본 6개를 … 핵심 변수들과 비교해서 그것이 맞는지 아닌지 뭐 알 수 없는지 등을 이제 분류」하는 물음입니다.

### 결정 — 여섯 줄 가운데 맞은 것은 「추우면 LNG가 는다」 하나이고, 「발전용이면 계절 영향이 난방만큼 크지 않을 것」은 반대였다. 나머지 넷(탄소 · 연료별 경제성 · 지정학 위험 · 상대 가격)은 모델 밖에 두어 여기서는 볼 수 없다

**같이 정한 것**

1. 핵심인자 다섯 가운데 여섯 줄에 없던 것 — 몇 시인가 · 햇빛 · 더운 쪽 기온은 놓쳤다. 여섯 줄을 적기 전인 09-13(옛 4단계 — 2023 · 2024년 데이터를 처음 그림과 표로 본 날)에 보고도 원인 여섯 줄에 넣지 않았다. 몇 시인가는 그날 「그나마 찾아볼 수 있는건 hour24, is_weekend 두 개의 값이 상대적으로 높은 값을 가지고 있는데 여기서 무언가를 찾아봐야 하지 않을까?」, 더운 쪽 기온은 「28~36도까지 기울기가 계속 오르는데」라고 적으셨고, 햇빛은 클로드가 꺼낸 이야기(낮에는 늘어난 태양광 때문에 LNG가 준다)로 나왔다. 그 가운데 몇 시인가가 다섯 가운데 가장 크다. 무슨 날인가 · 몇 월인가도 여섯 줄에는 없었지만, 여섯 줄을 적은 날(09-18) 데이터를 보고 이미 알고 있었다(1단계 물음 2의 「처음 쓴 것」 — 「주중인가 주말인가의 차이와 어떤 계절, 즉 월별로 사용되는 양의 차이가 존재했어」). 그래서 「짐작에 없었는데 나왔다」는 하나도 남지 않는다(창주 님 「놓쳤으면 놓쳤다고 적으면 될 것 같어.」 — 3-가 「✍️ 다시 읽고 나서」).
2. 몇 월인가는 ②(추위)에 붙이지 않는다. 모델의 몇 월인가는 기온 · 햇빛을 그대로 두고 달만 바꿨을 때 남는 차이라, 붙이면 추위를 두 번 센다(클로드 제안에 「너의 제안대로 가자」).
3. ②는 「추울수록 LNG 발전량이 늘었다」까지가 맞은 것이다. 왜 느는지(전기 난방으로 전력 수요가 늘어서인지, 열병합 발전소가 겨울에 난방열을 공급하면서 전기도 같이 만들어서인지)는 모델이 보여 주지 않았고, 우리가 짐작해 적는 해석이다.
4. 1단계에 적은 물음과 원인 여섯 줄은 고치지 않는다(클로드 판단 — 창주 님이 맡기셨고, 「적합하게 하자. 뭐 이정도면 되지 않을까?」로 확인하셨다 — 3-가 「✍️ 다시 읽고 나서」). 짐작이 빗나간 것이지 물음(2026년은 평소와 얼마나 달랐나)이 잘못된 것이 아니고, 볼 수 없는 넷은 일부러 모델 밖에 둔 것이라 9 · 10단계에서 쓴다. 고치면 11단계에 「처음에는 이렇게 짐작했는데 데이터로는 이랬다」를 적을 수 없다.
5. 여섯 줄은 2단계(09-12)와 옛 4단계(09-13)에서 데이터를 일부 본 뒤인 09-18에 적었다 — 데이터를 보기 전에 세운 가설이 아니다. ②의 추위도 09-13 기온 그림을 본 뒤에 적은 것이다(`docs/4단계_09-13_기록.md`). 창주 님도 09-18에 「이미 데이터를 본 상태로 진행하니 그건 어쩔 수 없는거지」라고 적으셨다. 11단계 결론에도 그렇게 적는다.

**이 물음에서 알게 된 것** — 처음 짐작 여섯 줄 가운데 모델로 맞았는지 확인할 수 있었던 것은 둘(추위 · 계절)뿐이었다. 나머지 넷은 일부러 모델 밖에 둔 것(석탄 · 가격 · LNG 수입량)에 대한 짐작이라, 틀린 것이 아니라 여기서는 볼 수 없는 것이다. 거꾸로 LNG를 가장 크게 움직인 몇 시인가와 햇빛 · 더운 쪽 기온은 09-13에 보고도 원인 여섯 줄에 넣지 않았다(놓쳤다).

**결정 근거** — 3-가 「✍️ 내 답변」 ① ~ ⑦과 「✍️ 다시 읽고 나서」(10-17 채팅 문답을 옮김), 그 아래 클로드 답변의 정리 표. 숫자는 이미 낸 셀에서 가져왔고 새로 돌린 코드는 없다.

- ② — 16도에서 영하 3.2도로 기온만 바꿔 넣으면 모델 넷 모두 LNG를 더 맞혔다(+3,776 ~ +5,401 MWh, 1-마 [표 2]).
- ③ — 실제 기록으로 1월 20,648 · 10월 15,225 MWh였고, 날씨를 그대로 두고 달만 바꾸면 17,672 · 17,121이었다(2-라 [표 3]).
- ⑦ — 24도보다 1도 더울 때마다 +661, 16도보다 1도 추울 때마다 +239 MWh(2-사 [표 1], 모델 넷의 평균). 09-13에 보고 적으신 것은 `docs/4단계_09-13_기록.md`에 있다.
- ① · ④ · ⑤ · ⑥ — 모델에 넣은 열(준비 ②)에 석탄 · 가격 · LNG 수입량이 없다. 모델 밖 숫자는 4단계 2-라 · 3-다 · 3-라와 7단계 3-하에서 가져왔다.
- 1단계를 고치지 않는 것은 3-가 「✍️ 읽고 나서」의 「가장 적합하게 ?? …」에 클로드가 답한 판단이다.

**남은 것**

- 9단계 — 핵심인자 다섯으로 2026년 1 ~ 6월의 「평소라면」을 맞힌다. ① · ⑤는 2026년 차이 옆에 같은 달 석탄 · 원전 발전량 변화를 놓고, 「석탄에서 가져올거라고 예상」(⑤, 10-17)이 맞는지 본다.
- 10단계 — ④ · ⑥(연료별 경제성 · 상대 가격)은 2026년 차이의 까닭 후보로 쓴다.
- 11단계 — 처음 짐작과 데이터(맞았다 하나 · 반대였다 하나 · 볼 수 없다 넷 · 놓친 것 셋 · 이미 알고 있던 것 둘)를 결론에 나란히 적는다. 「모델이 짐작에 없던 것을 찾아냈다」고 적지 않는다.
- 이 물음은 7단계에서 이미 본 2025년 결과(3-하)를 옮겨 적었고, 모델이나 입력을 고르는 데 쓰지 않았다. 2026년 LNG는 쓰지 않았다.

</div>

## 3-가 — 1단계 원인 여섯 줄을 하나씩, 물음 2에서 나온 것과 맞춰 보기

**이 셀이 답하는 물음** — 창주 님 한 줄의 전부입니다(「원인으로 본 6개를 … 핵심 변수들과 비교해서 그것이 맞는지 아닌지 뭐 알 수 없는지 등을 이제 분류」). 코드는 없습니다. 맞춰 볼 숫자는 물음 2에서 이미 나왔습니다.

**어떻게 적나** — 줄마다 물음 2에서 무엇이 나왔는지 찾아 적고, 그래서 무엇인지 적습니다. 중고차 값을 맞히는 모델이라면 이렇게 됩니다(10-17 채팅에서 든 예, 지어낸 것).

| 처음 짐작 | 모델에서 나온 것 | 적는 말 |
|---|---|---|
| 주행거리가 길수록 값이 내려간다 | 모델이 주행거리를 크게 썼고, 길수록 값이 내려갔다 | 맞았다 |
| 색이 밝을수록 비싸다 | 어두울수록 비쌌다 | 반대였다 |
| 브랜드가 값을 정한다 | 브랜드는 모델에 넣지 않았다 | 여기서는 볼 수 없다 |
| (짐작에 없음) | 모델이 연식을 가장 크게 썼다 | 짐작에 없었는데 나왔다 |

**맞춰 볼 재료 — 물음 2에서 나온 것**

모델에 넣은 다섯이 모두 핵심인자이고, 아래는 큰 순서입니다. 「더 틀린 양」은 그 변수의 값만 시각끼리 뒤섞었을 때 한 시각에 더 틀린 양입니다(뒤섞어 보기 2-다, 모델 넷의 평균). 「어느 쪽으로」는 그 변수만 바꿔 넣었을 때 모델이 맞힌 LNG가 어느 쪽으로 움직였는지입니다(그 변수만 바꿔 넣어 본 1-마 · 2-나).

| 순서 | 핵심인자 | 더 틀린 양 (한 시각 MWh) | 어느 쪽으로 |
|---|---|---|---|
| 1 | 몇 시인가 | 2,923 | 새벽에 적고, 늦은 오후 · 저녁에 많다 |
| 2 | 무슨 날인가 | 1,418 | 평일에 많고, 일요일에 적다 |
| 3 | 기온 | 1,002 | 16도보다 추울수록, 24도보다 더울수록 많다 |
| 4 | 햇빛 | 896 | 낮에 일사가 강할수록 적다 |
| 5 | 몇 월인가 | 280 | 날씨는 그대로 두고 달만 바꾸면, 움직인 폭이 다섯 가운데 가장 작다 |

모델에 넣지 않은 것 — 원전 · 석탄 · 가격 · 총 전력수요 · 습도 · 태양광 설비용량. 넣지 않은 까닭은 물음 2 결론 칸의 「넣지 않은 것과 그 까닭」 표에 있습니다.

여섯 줄은 1단계 결론 칸에 줄여 적은 것이고, 원문은 `01_과제정의`의 「✍️ 물음 2 — 내가 정한 것」에 있습니다.

# 8단계 한눈에 보기 — 자세한 판

맨 위 「8단계 한눈에 보기」 표의 자세한 판입니다(10-14). 창주 님은 맨 위 표만 보시면 되고, 이 판은 근거를 찾거나 다시 짚을 때 봅니다.

**이 칸은 왜 있나.** 10-13에 물음 2의 마지막 셀(2-바)을 보시다가 흐름을 놓치셔서, 1단계부터 채팅으로 다시 짚었습니다. 물음 2는 셀이 창주 님 물음보다 클로드가 정한 순서로 늘어서, 셀마다 왜 있는지가 보이지 않았습니다 — 여섯 셀 가운데 창주 님 물음에서 나온 것은 2-나 하나였습니다. 그래서 이 칸과 아래 두 칸에 물음마다 「물은 것 → 한 일 → 나온 것 → 정한 것」을, 셀마다 「답한 물음 · 누가 꺼냈나」를 모았습니다. 셀은 지우지 않았습니다. 숫자와 판단의 근거는 괄호 안에 적은 셀에 있습니다.

**한 줄로** — 8단계에서 정하는 것은 둘입니다. 9단계에서 2026년을 맞힐 모델 하나는 그래디언트 부스팅으로 정했습니다(물음 1). 핵심인자는 모델에 넣은 다섯 모두이고, 몇 시인가 → 무슨 날인가 → 기온 → 햇빛 → 몇 월인가 순서로 LNG를 움직였습니다(물음 2, 10-17에 끝냄). 1단계에 짐작한 원인 여섯 줄과 맞춰 보니 맞은 것은 추위 하나, 반대였던 것은 「발전용이라 계절 영향이 작을 것」 하나였고, 넷은 모델 밖에 두어 볼 수 없었습니다(물음 3, 10-17에 끝냄).

**지금 어디** — 물음 셋을 모두 끝냈고 개념 표 「내 말로」도 채웠습니다(10-17). 8단계는 10-17에 닫았고, 다음은 9단계입니다. 물음마다 결과와 결정은 맨 위 「8단계 한눈에 보기」에 있습니다.

**[표 1] 8단계의 앞과 뒤**

| | 무엇 |
|---|---|
| 앞 — 7단계(모델 넷을 만들어 2025년을 채점하고, 어디서 틀렸는지 본 단계)에서 정한 것 | 모델 넷(선형회귀 · 기온 × 시간대 선형회귀 · 랜덤포레스트 · 그래디언트 부스팅)이 모두 2025년을 기준선(모델 없이, 배운 해의 같은 달 · 같은 시각 · 같은 날 종류끼리 낸 평균)보다 잘 맞혀서 넷 다 쓰기로 했고, 하나 고르기는 8단계로 넘겼다. 고르는 기준의 첫 줄은 창주 님 말 「현실적으로 예측력이 높고 현실을 잘 반영하는 것」으로 두었다. 모델에 새 변수는 넣지 않기로 했다 — 2025년에 틀린 곳을 본 뒤에 변수를 넣으면, 그 변수는 2025년에 맞추려고 고른 것이라 2025년 점수가 처음 보는 해의 점수가 아니게 된다 |
| 8단계가 정하는 것 | 9단계에 가져갈 모델 하나(물음 1)와, 핵심인자 목록과 고른 까닭(물음 2)이다. 1단계에 적은 원인 목록과 맞는지도 확인한다(물음 3, 점검) |
| 뒤로 넘기는 것 | 9단계(2026년 1 ~ 6월이 평소와 얼마나 달랐나 보는 단계)에서 고른 모델을 같은 열 · 같은 설정으로 2023 ~ 2025년에 다시 배워 2026년을 맞히고, 실제와의 차이를 본다. 11단계(결론)에서 핵심인자와 그 까닭을 README에 적는다 |

**어느 프로젝트에서나 이 단계는** 모델 하나를 고르고, 어느 변수가 결과를 움직였는지 정합니다. 모델은 교재 순서대로 설명력(얼마나 잘 맞히나) → 안정성(나눠 봐도 꾸준한가) → 효율성(같으면 더 단순한 것)을 보고 고릅니다. 변수는 한 모델의 변수 중요도만으로 정하지 않고 여러 방법으로 본 것을 모아 정합니다. 마지막에 처음(1단계)에 짐작한 원인과 맞는지 봅니다.

## 물음마다 — 물은 것 → 한 일 → 나온 것 → 정한 것